In [1]:
from xarray_utils import analyze_netcdf, zarr_to_netcdf, find_missing_days
import pandas as pd
import xarray as xr
import numpy as np
import sklearn as sk
import sklearn as sk


In [2]:
# Open the Zarr dataset
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)

analyze_netcdf("../data/raw/IMD_rainfall_0p25.nc")


Analysis for NetCDF File: IMD_rainfall_0p25.nc

--- Dimensions ---
time: 31046
lat: 129
lon: 135

--- Coordinates ---
- lat:
    dtype: float64
    shape: (129,)
    attributes: {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (135,)
    attributes: {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- time:
    dtype: datetime64[ns]
    shape: (31046,)
    attributes: {'long_name': 'time', 'standard_name': 'time'}

--- Data Variables ---
- rain:
    dtype: float64
    shape: (31046, 129, 135)
    dimensions: ('time', 'lat', 'lon')
    attributes: {'long_name': 'Rainfall', 'units': 'mm/day'}

--- Global Attributes ---
Conventions: CF-1.7
comment: 
crs: epsg:4326
history: 2026-06-19 06:45:25.930728 Python
references: 
source: https://imdpune.gov.in/
title: IMD gridded data



In [3]:
# Open the Zarr dataset
ds_ecm = xr.open_zarr("../data/processed/s2s_reforecast_sorted.zarr")

analyze_netcdf("../data/raw/s2s_reforecast.nc")

Analysis for NetCDF File: s2s_reforecast.nc

--- Dimensions ---
time: 3720
step: 43
lat: 33
lon: 35

--- Coordinates ---
- lat:
    dtype: float64
    shape: (33,)
    attributes: {'long_name': 'latitude', 'standard_name': 'latitude', 'stored_direction': 'decreasing', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (35,)
    attributes: {'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- step:
    dtype: int64
    shape: (43,)
- time:
    dtype: datetime64[ns]
    shape: (3720,)
    attributes: {'long_name': 'initial time of forecast', 'standard_name': 'forecast_reference_time'}

--- Data Variables ---
- 10m_u_component_of_wind:
    dtype: float32
    shape: (3720, 43, 33, 35)
    dimensions: ('time', 'step', 'lat', 'lon')
    attributes: {'GRIB_NV': np.int64(0), 'GRIB_Nx': np.int64(35), 'GRIB_Ny': np.int64(33), 'GRIB_cfName': 'eastward_wind', 'GRIB_cfVarName': 'u10', 'GRIB_dataType': 'cf', 'GRIB_gridDefinitionDescription': 'Latitude/longi

In [4]:

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)


In [5]:
ds_ecmv.coords

Coordinates:
  * time        (time) datetime64[ns] 30kB 2005-01-01 2005-01-03 ... 2024-12-31
  * step        (step) int64 344B 0 1 2 3 4 5 6 7 8 ... 35 36 37 38 39 40 41 42
    valid_time  (time, step) datetime64[ns] 1MB 2005-01-01 ... 2025-02-11
  * lat         (lat) float64 264B 38.5 37.5 36.5 35.5 34.5 ... 9.5 8.5 7.5 6.5
  * lon         (lon) float64 280B 66.5 67.5 68.5 69.5 ... 97.5 98.5 99.5 100.5

In [6]:
train = ds_ecmv.sel(time=slice("2005-01-01", "2019-12-31"))
val = ds_ecmv.sel(time=slice("2020-01-01", "2021-12-31"))
test = ds_ecmv.sel(time=slice("2022-01-01", "2024-12-31"))

In [8]:
"""
Combines ECMWF predictors (coarse, e.g. 1deg) and IMD rain (target, 0.25deg)
into a single Dataset shaped like ECMWF: dims (time, step, lat, lon), with
IMD rain attached as a new data_var aligned via valid_time.

Two separate operations, NOT a blind xr.merge:
  1. Regrid ECMWF onto IMD's lat/lon grid (coarse -> fine, standard for
     downscaling: predictors get put on the target's pixel grid).
  2. Attach IMD rain by exact valid_time date match (NOT nearest-neighbor
     in time -- exact match avoids silently pairing a forecast step with
     the wrong day's observed rain).

ASSUMES:
  ecmwf_ds: dims (time, step, lat, lon), lat/lon ~1deg spacing
  imd_ds:   dims (time, lat, lon), lat/lon 0.25deg spacing, daily
"""

import numpy as np
import xarray as xr

IMD_TARGET_VAR = "rain"          # actual IMD rainfall variable name -- edit
NEW_TARGET_NAME = "target_rain"     # name it'll be attached as in the combined ds


def ensure_valid_time(ds):
    if "valid_time" not in ds.coords:
        ds = ds.assign_coords(
            valid_time=ds["time"] + ds["step"].astype("timedelta64[D]")
        )
    return ds


def regrid_ecmwf_to_imd(ecmwf_ds, imd_ds, method="linear"):
    """
    Interpolates every ECMWF variable onto IMD's lat/lon grid.
    method='linear' = bilinear-equivalent, standard for coarse->fine
    predictor regridding (not conservative -- conservative matters more
    when aggregating fine->coarse, which isn't what's happening here).
    """
    return ecmwf_ds.interp(lat=imd_ds["lat"], lon=imd_ds["lon"], method=method)


def attach_imd_target(ecmwf_ds, imd_ds, imd_var=IMD_TARGET_VAR, new_name=NEW_TARGET_NAME):
    """
    Attaches IMD rain to ecmwf_ds as a new (time, step, lat, lon) variable,
    matched to each forecast's valid_time by EXACT date (floor to day).
    Missing IMD dates become NaN, not a silently wrong nearest date.
    """
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    valid_dates = ecmwf_ds["valid_time"].dt.floor("D")  # dims: (time, step)

    imd_var_da = imd_ds[imd_var]
    imd_dates = imd_var_da["time"].dt.floor("D").values
    assert len(set(imd_dates)) == len(imd_dates), "IMD time index has duplicate dates"

    flat_dates = valid_dates.values.ravel()
    n_missing = (~np.isin(flat_dates, imd_dates)).sum()
    if n_missing:
        print(f"WARNING: {n_missing}/{len(flat_dates)} valid_time dates have no IMD match -> NaN")

    # exact-match reindex: unmatched dates become NaN rather than nearest-neighbor guesses
    imd_reindexed = imd_var_da.reindex(time=flat_dates)


    target_values = imd_reindexed.values.reshape(
        valid_dates.shape + imd_var_da.shape[1:]  # (time, step, lat, lon)
    )
    target_da = xr.DataArray(
        target_values,
        dims=("time", "step", "lat", "lon"),
        coords={
            "time": ecmwf_ds["time"],
            "step": ecmwf_ds["step"],
            "lat": ecmwf_ds["lat"],
            "lon": ecmwf_ds["lon"],
        },
    )
    ecmwf_ds = ecmwf_ds.copy()
    ecmwf_ds[new_name] = target_da
    return ecmwf_ds


if __name__ == "__main__":
    # --- example wiring, replace with your actual datasets ---
    ecmwf_ds = ds_ecmv
    imd_ds = ds_imd

    ecmwf_regridded = regrid_ecmwf_to_imd(ecmwf_ds, imd_ds, method="linear")
    combined = attach_imd_target(ecmwf_regridded, imd_ds)

    print(combined)
    print(f"\n{NEW_TARGET_NAME} NaN fraction: "
          f"{float(combined[NEW_TARGET_NAME].isnull().mean().values):.3%}")

    # `combined` is now a drop-in `ds` for the earlier xarray baseline script --
    # set TARGET_VAR = "tp_obs" there and you're ready to run climatology /
    # lead-stratified linear regression on it.

<xarray.Dataset> Size: 256GB
Dimensions:                   (time: 3720, step: 43, lat: 129, lon: 135)
Coordinates:
  * time                      (time) datetime64[ns] 30kB 2005-01-01 ... 2024-...
  * step                      (step) int64 344B 0 1 2 3 4 5 ... 38 39 40 41 42
    valid_time                (time, step) datetime64[ns] 1MB 2005-01-01 ... ...
  * lat                       (lat) float64 1kB 6.5 6.75 7.0 ... 38.0 38.25 38.5
  * lon                       (lon) float64 1kB 66.5 66.75 67.0 ... 99.75 100.0
Data variables: (12/22)
    10m_u_component_of_wind   (time, step, lat, lon) float32 11GB dask.array<chunksize=(20, 43, 129, 135), meta=np.ndarray>
    10m_v_component_of_wind   (time, step, lat, lon) float32 11GB dask.array<chunksize=(20, 43, 129, 135), meta=np.ndarray>
    2m_temperature            (time, step, lat, lon) float32 11GB dask.array<chunksize=(20, 43, 129, 135), meta=np.ndarray>
    mean_sea_level_pressure   (time, step, lat, lon) float32 11GB dask.array<chunksize=

In [ ]:
# """
# Raw-value baseline (NO climatology, NO anomalies) for S2S precipitation
# forecasting, lead-stratified linear regression across lead 0-42 (step).

# Purpose: deliberately the "naive" version -- predicting raw precipitation
# directly from raw predictor values, per lead day -- so you have a concrete
# number to compare against once you add climatology/anomaly preprocessing.
# Expect reasonable skill at short leads and collapse (at or below the
# climatology floor) by weeks 3-6. That gap IS the argument for anomaly-based
# modeling.

# KEY RESTRUCTURE vs. the first draft
# -----------------------------------
# The model only ever consumes SPATIAL MEANS. So we reduce to spatial means
# as early as possible, in ONE lazy graph, computed ONCE:

#   old:  interp to 0.25 (36x expansion) -> replicate IMD 43x (~6 GB)
#         -> loop over 43 steps x N vars, each triggering a full recompute
#   new:  build mask+weights -> lazy interp -> lazy weighted mean
#         -> single dask.compute() -> ~7 MB of numpy -> everything else instant

# Net effect: ~86*N redundant interpolations collapse to 1, and peak memory
# drops from GBs to MBs. There is nothing left worth parallelising afterwards
# (the 43 LinearRegression fits are microseconds each).

# Pipeline:
#   1. Derive IMD valid-cell mask from the data (NOT a land-sea mask -- IMD's
#      domain is the gauge-interpolation footprint, ~India's political boundary)
#   2. Reduce ECMWF (masked, area-weighted) and IMD to spatial-mean series
#   3. Align via valid_time = init_time + step (exact date match)
#   4. Split by initialization year (train/val/test)
#   5. Fit one LinearRegression per lead (step), RAW values, no anomaly step
#   6. Evaluate RMSE + correlation + skill vs. a train-mean climatology floor
# """

# import time
# from contextlib import contextmanager

# import dask
# import numpy as np
# import xarray as xr
# from dask.diagnostics import ProgressBar
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import root_mean_squared_error

# # ---- CONFIG ----
# IMD_TARGET_VAR = "rain"
# NEW_TARGET_NAME = "target_rain"
# MIN_SAMPLES_PER_LEAD = 20

# # Chunking for the lazy interp. lat/lon MUST be single chunks (interp needs
# # the full spatial extent per chunk); parallelism comes from the time dim.
# # Tune TIME_CHUNK down if you see memory pressure on a 16 GB M4.
# TIME_CHUNK = 40


# # ---------------- progress plumbing ----------------

# @contextmanager
# def stage(name):
#     """Timed stage marker so you can see where the wall clock actually goes."""
#     print(f"[ ] {name} ...", flush=True)
#     t0 = time.perf_counter()
#     yield
#     print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# # ---------------- Step 1: mask + weights ----------------

# def build_imd_mask(imd_ds, var=IMD_TARGET_VAR, verbose=True):
#     """
#     Valid-cell mask derived from IMD itself.

#     IMPORTANT: this is NOT a land-sea mask. IMD 0.25 is the interpolation
#     domain of India's rain-gauge network, so Pakistan/Nepal/Bangladesh/
#     Sri Lanka are NaN despite being land. Never build this from ERA5 lsm.

#     Also reports whether missingness is static -- if the valid-cell count
#     varies day to day you have station dropouts and need a per-sample mask.
#     """
#     da = imd_ds[var]
#     mask = da.notnull().any(dim="time")

#     if verbose:
#         n_valid = int(mask.sum())
#         n_total = int(mask.size)
#         print(f"    mask: {n_valid}/{n_total} cells valid ({100 * n_valid / n_total:.1f}%)")

#         # cheap static-ness check on a subsample (full check is expensive)
#         sub = da.isel(time=slice(0, None, max(1, da.sizes["time"] // 200)))
#         per_day = sub.notnull().sum(dim=["lat", "lon"]).compute()
#         lo, hi = int(per_day.min()), int(per_day.max())
#         if lo != hi:
#             print(f"    WARNING: valid-cell count varies {lo}..{hi} across days "
#                   f"-> missingness is NOT static; consider a per-sample mask")
#         else:
#             print(f"    mask is static ({lo} cells/day)")

#     return mask


# def make_weights(mask):
#     """cos(lat) area weighting, zeroed outside the IMD domain.

#     Fixes two things at once:
#       - unweighted .mean() over 6.5-38.5N mis-weights cells by ~20% end to end
#       - restricts the ECMWF predictor mean to the SAME region as the IMD
#         target mean (otherwise you regress ocean-inclusive predictors onto
#         land-only targets)
#     """
#     w = np.cos(np.deg2rad(mask["lat"]))
#     return (w * mask).fillna(0.0)


# def spatial_mean(obj, weights):
#     """Masked, area-weighted spatial mean. Lazy if obj is dask-backed."""
#     return obj.weighted(weights).mean(dim=["lat", "lon"], skipna=True)


# # ---------------- Step 2: the one expensive compute ----------------

# def ensure_valid_time(ds):
#     if "valid_time" in ds.coords:
#         return ds
#     step = ds["step"]
#     if not np.issubdtype(step.dtype, np.timedelta64):
#         raise TypeError(
#             f"'step' has dtype {step.dtype}, expected timedelta64. "
#             "Blind .astype('timedelta64[D]') on an int step silently produces "
#             "nonsense valid_times -- fix the coordinate upstream instead."
#         )
#     return ds.assign_coords(valid_time=ds["time"] + step)


# def reduce_to_series(ecmwf_ds, imd_ds, time_chunk=TIME_CHUNK):
#     """
#     Collapse both datasets to spatial-mean series in a single dask compute.

#     Returns
#     -------
#     feat : xr.Dataset  dims (time, step), one var per predictor  -- ~MBs
#     imd  : xr.DataArray dims (time,), daily all-India mean rain   -- ~KBs
#     """
#     mask = build_imd_mask(imd_ds)
#     w = make_weights(mask)

#     # Rechunk so interp gets whole lat/lon planes; time is the parallel axis.
#     if ecmwf_ds.chunks:
#         ecmwf_ds = ecmwf_ds.chunk({"time": time_chunk, "lat": -1, "lon": -1})

#     # Lazy: interp -> mask+weight -> reduce. The 0.25 grid is never materialised
#     # beyond a single transient chunk at a time.
#     ecm_fine = ecmwf_ds.interp(lat=imd_ds["lat"], lon=imd_ds["lon"], method="linear")
#     feat = spatial_mean(ecm_fine, w)

#     # Reduce IMD to a 1-D daily series BEFORE any (time, step) expansion.
#     # The old code replicated the full 3-D field 43x (~6 GB); this is ~KBs.
#     imd = spatial_mean(imd_ds[IMD_TARGET_VAR], w)

#     print("    computing spatial means (single pass, threaded across cores)...")
#     with ProgressBar():
#         feat, imd = dask.compute(feat, imd)

#     return feat, imd


# # ---------------- Step 3: align target via valid_time ----------------

# def attach_target(feat, imd, new_name=NEW_TARGET_NAME):
#     """Map each (init, step) -> IMD spatial mean on the valid date.

#     Now a 1-D lookup instead of a 4-D reindex+reshape.
#     """
#     feat = ensure_valid_time(feat)

#     imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
#     dates = imd["time"].values
#     if len(np.unique(dates)) != len(dates):
#         raise ValueError("IMD time index has duplicate dates after floor('D')")

#     vt = feat["valid_time"].dt.floor("D")
#     flat = vt.values.ravel()

#     n_missing = int((~np.isin(flat, dates)).sum())
#     if n_missing:
#         print(f"    WARNING: {n_missing}/{len(flat)} valid_times have no IMD "
#               f"match -> NaN target ({100 * n_missing / len(flat):.1f}%)")

#     tgt = imd.reindex(time=flat).values.reshape(vt.shape)
#     feat = feat.copy()
#     feat[new_name] = (("time", "step"), tgt)
#     return feat


# # ---------------- Step 4: split by init year ----------------

# def split_by_year(ds, test_years, val_years=None):
#     years = ds["time"].dt.year
#     test_mask = years.isin(list(test_years))
#     val_mask = years.isin(list(val_years)) if val_years else xr.zeros_like(test_mask, dtype=bool)
#     train_mask = ~(test_mask | val_mask)
#     return ds.sel(time=train_mask), ds.sel(time=val_mask), ds.sel(time=test_mask)


# # ---------------- Step 5-6: fit + evaluate ----------------

# def get_feature_vars(ds, target_var):
#     return [v for v in ds.data_vars if v != target_var]


# def step_to_xy(ds, step, target_var, feature_vars, col_means=None):
#     """Pure numpy now -- no dask, no recompute. Microseconds."""
#     s = ds.sel(step=step)
#     X = np.column_stack([s[v].values for v in feature_vars]).astype(float)
#     y = np.asarray(s[target_var].values, dtype=float)

#     valid = np.isfinite(y)
#     X, y = X[valid], y[valid]

#     if col_means is None:
#         col_means = np.nanmean(X, axis=0) if len(X) else np.zeros(X.shape[1])
#         col_means = np.where(np.isnan(col_means), 0.0, col_means)

#     rows, cols = np.where(np.isnan(X))
#     X[rows, cols] = col_means[cols]
#     return X, y, col_means


# def train_lead_stratified_raw(train_ds, target_var, feature_vars,
#                               min_samples=MIN_SAMPLES_PER_LEAD):
#     models = {}
#     steps = train_ds["step"].values
#     for i, step in enumerate(steps):
#         X, y, col_means = step_to_xy(train_ds, step, target_var, feature_vars)
#         if len(X) < min_samples:
#             print(f"    step {i:>3}: skipped ({len(X)} < {min_samples} samples)")
#             continue
#         m = LinearRegression().fit(X, y)
#         # climatology floor: what a flat train-mean predictor would give.
#         # Without this, "RMSE = 4.2" is uninterpretable.
#         models[i] = {"model": m, "col_means": col_means, "train_mean": float(y.mean())}
#     print(f"    fitted {len(models)}/{len(steps)} leads")
#     return models


# def evaluate_lead_stratified_raw(eval_ds, models, target_var, feature_vars):
#     results = {}
#     for step_idx, m in sorted(models.items()):
#         step = eval_ds["step"].values[step_idx]
#         X, y, _ = step_to_xy(eval_ds, step, target_var, feature_vars,
#                              col_means=m["col_means"])
#         if len(X) == 0:
#             continue
#         y_pred = m["model"].predict(X)
#         rmse = root_mean_squared_error(y, y_pred)
#         rmse_clim = root_mean_squared_error(y, np.full_like(y, m["train_mean"]))
#         results[step_idx] = {
#             "rmse": rmse,
#             "rmse_clim": rmse_clim,
#             "skill": 1.0 - rmse / rmse_clim if rmse_clim > 0 else np.nan,
#             "corr": np.corrcoef(y, y_pred)[0, 1] if len(y) > 1 and y.std() > 0 else np.nan,
#             "n": len(y),
#         }
#     return results


# # ---------------- driver ----------------

# if __name__ == "__main__":
#     ecmwf_ds = ds_ecmv   # noqa: F821  <- replace
#     imd_ds = ds_imd      # noqa: F821  <- replace

#     t_start = time.perf_counter()

#     with stage("Reducing to spatial-mean series (the only expensive step)"):
#         feat, imd = reduce_to_series(ecmwf_ds, imd_ds)
#         mb = sum(v.nbytes for v in feat.data_vars.values()) / 1e6
#         print(f"    feature series: {dict(feat.sizes)}  ({mb:.1f} MB in memory)")

#     with stage("Attaching IMD target via valid_time"):
#         combined = attach_target(feat, imd)

#     with stage("Splitting by init year"):
#         feature_vars = get_feature_vars(combined, NEW_TARGET_NAME)
#         all_years = sorted(set(combined["time"].dt.year.values.tolist()))
#         test_years = set(all_years[-3:])
#         val_years = set(all_years[-6:-3])
#         train_ds, val_ds, test_ds = split_by_year(combined, test_years, val_years)
#         print(f"    train {train_ds.sizes['time']} | "
#               f"val {val_ds.sizes['time']} | test {test_ds.sizes['time']} inits")
#         print(f"    val years {sorted(val_years)} | test years {sorted(test_years)}")
#         print(f"    features ({len(feature_vars)}): {feature_vars}")

#     with stage("Fitting lead-stratified linear regressions"):
#         models = train_lead_stratified_raw(train_ds, NEW_TARGET_NAME, feature_vars)

#     with stage("Evaluating on validation"):
#         scores = evaluate_lead_stratified_raw(val_ds, models, NEW_TARGET_NAME, feature_vars)

#     print(f"\nTotal: {time.perf_counter() - t_start:.1f}s")
#     print("\nRAW-VALUE linear regression (no climatology/anomaly)")
#     print("validation skill by lead. skill > 0 means it beats a flat train-mean.\n")
#     print(f"{'lead':>5} {'rmse':>9} {'rmse_clim':>10} {'skill':>8} {'corr':>8} {'n':>6}")
#     for step_idx, m in scores.items():
#         print(f"{step_idx:>5} {m['rmse']:>9.3f} {m['rmse_clim']:>10.3f} "
#               f"{m['skill']:>8.3f} {m['corr']:>8.3f} {m['n']:>6}")

[ ] Reducing to spatial-mean series (the only expensive step) ...
    mask: 4964/17415 cells valid (28.5%)
    mask is static (4964 cells/day)
    computing spatial means (single pass, threaded across cores)...
[########################################] | 100% Completed | 465.27 s
    feature series: {'step': 43, 'time': 3720}  (26.9 MB in memory)
[x] Reducing to spatial-mean series (the only expensive step)  (467.7s)
[ ] Attaching IMD target via valid_time ...
[x] Attaching IMD target via valid_time  (0.3s)
[ ] Splitting by init year ...
    train 2604 | val 558 | test 558 inits
    val years [2019, 2020, 2021] | test years [2022, 2023, 2024]
    features (21): ['10m_u_component_of_wind', '10m_v_component_of_wind', '2m_temperature', 'mean_sea_level_pressure', 'specific_humidity_1000', 'specific_humidity_200', 'specific_humidity_500', 'specific_humidity_850', 'temperature_1000', 'temperature_200', 'temperature_500', 'temperature_850', 'total_precipitation', 'u_component_of_wind_1000', 

/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_47627/3424544042.py:221: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(X, axis=0) if len(X) else np.zeros(X.shape[1])


In [ ]:
"""
S2S baseline suite for lead 0-42: day-of-year climatology, raw-value
lead-stratified linear regression, and anomaly-space lead-stratified linear
regression -- all scored on the SAME quantity against the SAME reference.

Supersedes the separate raw-baseline script: all three arms share one dask
reduction, so the whole thing runs in seconds after the initial compute.


WHY THREE ARMS
--------------
  clim   : DOY climatology. The honest floor. "Knowing what date it is."
  raw    : regress raw y on raw X. Expect it to sit AT the clim floor by
           ~lead 10-14, because raw regression mostly relearns the seasonal
           cycle -- which never decays with lead, hence the flat skill
           plateau you saw against a flat-annual-mean reference.
  anom   : regress y-anomaly on X-anomaly, seasonal cycle removed from both
           sides. Whatever skill survives here is real.

skill = 1 - rmse/rmse_clim, so skill > 0 means "beats knowing the date".
For the raw arm this is the number that matters; a flat annual mean is not
a defensible reference.


THE TWO CLIMATOLOGIES ARE DIFFERENT OBJECTS
-------------------------------------------
  clim_obs(doy)        -- observed. IMD rain on 15 July has ONE climatology
                          regardless of which lead reached that date.
                          Carrying a `step` dim here fragments samples ~43x
                          for zero information.
  clim_mod(doy, lead)  -- model. MUST be lead-dependent: the IFS drifts
                          toward its own attractor as lead grows, so its
                          bias at day 7 is not its bias at day 35
                          (Vitart 2004; Weigel et al. 2008).
That asymmetry IS the drift correction. Subtract each side from its own
climatology and the drift is gone before the regression sees anything.

Both are fit on TRAIN INITS ONLY. Computing climatology over all years
inflates reported skill (climpred calls this the "unfair" split).
"""

import time
from contextlib import contextmanager

import dask
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar
from sklearn.linear_model import LinearRegression

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
TARGET_NAME = "target_rain"
CLIM_WINDOW_DAYS = 3        # 3-day climatology window
STEP_UNITS = "auto"         # "auto" | "D" | "h" | "s" -- see normalize_step
MIN_SAMPLES_PER_LEAD = 20
MIN_CLIM_SAMPLES = 15       # warn if a (doy, lead) bin is thinner than this
TIME_CHUNK = 40

# ECMWF fluxes are accumulated from forecast start (GRIB stepType="accum").
# tp at step 42 is 42-DAY TOTAL RAIN, not day-42 rain. Auto-detected below.
CANDIDATE_ACCUM_VARS = ("tp", "ssr", "str", "ssrd", "strd", "sshf", "slhf",
                        "e", "ro", "sf", "cp", "lsp")


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- mask, weights, reduction ----------------

def build_imd_mask(imd_ds, var=IMD_TARGET_VAR, verbose=True):
    """Valid-cell mask from IMD itself -- NOT a land-sea mask. IMD 0.25 is the
    gauge-network interpolation footprint (~India's political boundary), so
    Pakistan/Nepal/Bangladesh are NaN despite being land."""
    da = imd_ds[var]
    mask = da.notnull().any(dim="time")
    if verbose:
        n, tot = int(mask.sum()), int(mask.size)
        print(f"    mask: {n}/{tot} cells valid ({100 * n / tot:.1f}%)")
    return mask


def make_weights(mask):
    """cos(lat) area weights, zeroed outside IMD's domain.

    Fixes two things: unweighted .mean() mis-weights cells by ~20% across
    6.5-38.5N, and -- more importantly -- restricts the ECMWF predictor mean
    to the SAME region as the IMD target mean. Without the mask you regress
    an ocean-inclusive predictor mean onto a land-only target mean.
    """
    return (np.cos(np.deg2rad(mask["lat"])) * mask).fillna(0.0)


def spatial_mean(obj, weights):
    return obj.weighted(weights).mean(dim=["lat", "lon"], skipna=True)


def normalize_step(ds, units=STEP_UNITS, verbose=True):
    """Coerce `step` to timedelta64, however it arrived.

    xarray encodes timedelta64 coords in zarr as ints + a units attr. Reading
    back without decode_timedelta=True gives the bare int, so `step` is often
    int64 (days OR hours OR seconds). cfgrib gives timedelta64[ns]. Handle both.

    The alternative -- your original ds["step"].astype("timedelta64[D]") --
    is silently catastrophic if step is int HOURS: 1008 hours becomes 1008
    DAYS and every valid_time is ~3 years off.
    """
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    if not np.issubdtype(v.dtype, np.number):
        raise TypeError(f"'step' has dtype {v.dtype}; expected timedelta64 or numeric.")

    u = units
    if u == "auto":
        mx = int(np.nanmax(v))
        u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
        if verbose:
            print(f"    step is {v.dtype} (max {mx}) -> reading as units='{u}'. "
                  f"If wrong, set STEP_UNITS explicitly.")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    """Normalise step, then build/verify valid_time = time + step.

    Verifies rather than trusting an existing valid_time: if it was built
    upstream with a bad unit assumption it is wrong, and a wrong valid_time
    is invisible -- it just quietly pairs forecasts with the wrong day's rain.
    """
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]

    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")

    return ds.assign_coords(valid_time=expected)


def step_days(ds):
    """Lead in whole days, whatever step was originally encoded as."""
    return (ds["step"].values / np.timedelta64(1, "D")).astype(int)


def reduce_to_series(ecmwf_ds, imd_ds, time_chunk=TIME_CHUNK):
    """Collapse both datasets to spatial-mean series in ONE dask compute.

    The model only consumes spatial means, so the 0.25 grid is never
    materialised beyond one transient chunk. Old build_climatology made 366
    full passes over a (time, step, lat, lon) array; here everything
    downstream is a few MB of numpy.
    """
    mask = build_imd_mask(imd_ds)
    w = make_weights(mask)

    if ecmwf_ds.chunks:
        ecmwf_ds = ecmwf_ds.chunk({"time": time_chunk, "lat": -1, "lon": -1})

    ecm_fine = ecmwf_ds.interp(lat=imd_ds["lat"], lon=imd_ds["lon"], method="linear")
    feat = spatial_mean(ecm_fine, w)
    imd = spatial_mean(imd_ds[IMD_TARGET_VAR], w)

    print("    computing spatial means (single pass, threaded)...")
    with ProgressBar():
        feat, imd = dask.compute(feat, imd)
    return feat, imd


# ---------------- de-accumulation ----------------

def detect_accumulated(feat, candidates=CANDIDATE_ACCUM_VARS, verbose=True):
    """Flag vars that increase monotonically in step -> accumulated from t0."""
    found = []
    for v in feat.data_vars:
        if v not in candidates:
            continue
        s = feat[v].isel(time=0).values
        s = s[np.isfinite(s)]
        if len(s) < 3:
            continue
        if np.mean(np.diff(s) >= -1e-9) > 0.95:
            found.append(v)
    if verbose:
        if found:
            print(f"    ACCUMULATED (will de-accumulate): {found}")
            print(f"    -> tp at step 42 was 42-day total rain, not day-42 rain")
        else:
            print("    no accumulated vars detected; leaving steps as-is")
    return found


def deaccumulate(feat, accum_vars):
    """diff along step. Commutes with interp and spatial mean (all linear),
    so doing it here on the reduced series is exact and ~free.

    After this, step k = flux accumulated DURING day k. Step 0 drops out --
    it was identically zero for accumulated vars, which is why lead 0 looked
    anomalously bad in the raw run.
    """
    if not accum_vars:
        return feat
    out = {}
    for v in feat.data_vars:
        out[v] = feat[v].diff("step") if v in accum_vars else feat[v].isel(step=slice(1, None))
    ds = xr.Dataset(out)
    for c in ("valid_time",):
        if c in feat.coords:
            ds = ds.assign_coords({c: feat[c].isel(step=slice(1, None))})
    return ds


# ---------------- target attach + split ----------------

def attach_target(feat, imd, new_name=TARGET_NAME):
    feat = ensure_valid_time(feat)
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    dates = imd["time"].values
    if len(np.unique(dates)) != len(dates):
        raise ValueError("IMD time index has duplicate dates after floor('D')")

    vt = feat["valid_time"].dt.floor("D")
    flat = vt.values.ravel()
    n_missing = int((~np.isin(flat, dates)).sum())
    if n_missing:
        print(f"    WARNING: {n_missing}/{len(flat)} valid_times unmatched "
              f"({100 * n_missing / len(flat):.1f}%) -> NaN target")

    tgt = imd.reindex(time=flat).values.reshape(vt.shape)
    feat = feat.copy()
    feat[new_name] = (("time", "step"), tgt)
    return feat


def split_by_year(ds, test_years, val_years=None):
    """Split on INIT date. Splitting on valid_time would leak one init's
    0-42 day window across train and test."""
    years = ds["time"].dt.year
    test_mask = years.isin(list(test_years))
    val_mask = years.isin(list(val_years)) if val_years else xr.zeros_like(test_mask, dtype=bool)
    train_mask = ~(test_mask | val_mask)
    return ds.sel(time=train_mask), ds.sel(time=val_mask), ds.sel(time=test_mask)


# ---------------- climatology (vectorised) ----------------

def _doy_clim(values, doys, window, n_doy=366):
    """Circular-window DOY mean. values/doys are 1-D, same length.
    Returns (n_doy,) means and (n_doy,) sample counts.

    Vectorised: builds one (366, n_samples) boolean matrix instead of the
    old 366-iteration loop over a 4-D array.
    """
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    m = np.minimum(d, n_doy - d) <= window
    m = m & np.isfinite(values)[None, :]
    counts = m.sum(axis=1)
    sums = (m * np.nan_to_num(values)[None, :]).sum(axis=1)
    return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan), counts


def build_obs_climatology(train_ds, target_var=TARGET_NAME, window=CLIM_WINDOW_DAYS):
    """clim_obs(doy) -- NO step dim.

    Built from UNIQUE valid dates reachable from train inits: dedup avoids
    over-weighting dates hit by several inits, and restricting to train-init
    reach avoids leaking val/test observations into the reference.
    """
    dates = train_ds["valid_time"].dt.floor("D").values.ravel()
    vals = np.asarray(train_ds[target_var].values).ravel()
    _, idx = np.unique(dates, return_index=True)
    dates, vals = dates[idx], vals[idx]
    doys = xr.DataArray(dates, dims="t").dt.dayofyear.values

    clim, counts = _doy_clim(vals, doys, window)
    thin = int((counts[counts > 0] < MIN_CLIM_SAMPLES).sum())
    print(f"    clim_obs: {len(dates)} unique dates, "
          f"median {int(np.median(counts[counts > 0]))} samples/doy-bin"
          + (f"  WARNING: {thin} bins < {MIN_CLIM_SAMPLES}" if thin else ""))
    return clim


def build_model_climatology(train_ds, feature_vars, window=CLIM_WINDOW_DAYS):
    """clim_mod(doy, lead) per predictor -- lead-dependent BY DESIGN.

    This is the drift correction: the IFS relaxes toward its own attractor
    as lead grows, so each lead needs its own climatology. Mirrors ECMWF's
    operational recipe (mean hindcast error per lead, pooled over nearby
    calendar dates).
    """
    doy_2d = train_ds["valid_time"].dt.dayofyear.values     # (time, step)
    out = {}
    min_count = np.inf
    for v in feature_vars:
        vals_2d = np.asarray(train_ds[v].values)            # (time, step)
        per_step = np.empty((366, vals_2d.shape[1]))
        for j in range(vals_2d.shape[1]):
            per_step[:, j], counts = _doy_clim(vals_2d[:, j], doy_2d[:, j], window)
            min_count = min(min_count, counts[counts > 0].min() if (counts > 0).any() else 0)
        out[v] = per_step
    print(f"    clim_mod: {len(feature_vars)} vars x 366 doy x "
          f"{vals_2d.shape[1]} leads, min {int(min_count)} samples/bin"
          + (f"  WARNING: thin bins, consider window > {window}"
             if min_count < MIN_CLIM_SAMPLES else ""))
    return out


# ---------------- design matrices ----------------

def step_xy(ds, j, feature_vars, target_var=TARGET_NAME,
            clim_obs=None, clim_mod=None, col_means=None):
    """Pure numpy. Returns X, y for lead index j.

    If clim_obs/clim_mod given, returns ANOMALIES: each side minus its own
    climatology (obs by doy, model by doy AND lead).
    """
    s = ds.isel(step=j)
    doy = s["valid_time"].dt.dayofyear.values
    X = np.column_stack([s[v].values for v in feature_vars]).astype(float)
    y = np.asarray(s[target_var].values, dtype=float)

    if clim_obs is not None:
        y = y - clim_obs[doy - 1]
        X = X - np.column_stack([clim_mod[v][doy - 1, j] for v in feature_vars])

    valid = np.isfinite(y)
    X, y = X[valid], y[valid]

    if col_means is None:
        col_means = np.nanmean(X, axis=0) if len(X) else np.zeros(X.shape[1])
        col_means = np.where(np.isnan(col_means), 0.0, col_means)
    r, c = np.where(np.isnan(X))
    X[r, c] = col_means[c]
    return X, y, col_means


def clim_reference(ds, j, clim_obs, target_var=TARGET_NAME):
    """Observed values and the DOY-climatology prediction for lead index j."""
    s = ds.isel(step=j)
    doy = s["valid_time"].dt.dayofyear.values
    y = np.asarray(s[target_var].values, dtype=float)
    p = clim_obs[doy - 1]
    m = np.isfinite(y) & np.isfinite(p)
    return y[m], p[m]


def _rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def _corr(a, b):
    return float(np.corrcoef(a, b)[0, 1]) if len(a) > 1 and a.std() > 0 and b.std() > 0 else np.nan


# ---------------- run all three arms ----------------

def run_baselines(train_ds, eval_ds, feature_vars, clim_obs, clim_mod,
                  min_samples=MIN_SAMPLES_PER_LEAD):
    results = {}
    n_steps = train_ds.sizes["step"]
    leads = step_days(train_ds)

    for j in range(n_steps):
        lead = int(leads[j])

        Xtr, ytr, cm = step_xy(train_ds, j, feature_vars)
        if len(Xtr) < min_samples:
            continue
        Xev, yev, _ = step_xy(eval_ds, j, feature_vars, col_means=cm)
        if len(Xev) == 0:
            continue

        # arm 1: DOY climatology -- the common reference
        y_c, p_c = clim_reference(eval_ds, j, clim_obs)
        rmse_clim = _rmse(y_c, p_c)

        # arm 2: raw LR
        raw_pred = LinearRegression().fit(Xtr, ytr).predict(Xev)
        rmse_raw = _rmse(yev, raw_pred)

        # arm 3: anomaly LR
        Xtr_a, ytr_a, cm_a = step_xy(train_ds, j, feature_vars,
                                     clim_obs=clim_obs, clim_mod=clim_mod)
        Xev_a, yev_a, _ = step_xy(eval_ds, j, feature_vars, clim_obs=clim_obs,
                                  clim_mod=clim_mod, col_means=cm_a)
        anom_pred = LinearRegression().fit(Xtr_a, ytr_a).predict(Xev_a)
        rmse_anom = _rmse(yev_a, anom_pred)
        # anomalies are already clim-relative, so rmse_clim in anomaly space
        # is just the anomaly std (predicting zero). Same reference either way.
        rmse_clim_a = _rmse(yev_a, np.zeros_like(yev_a))

        results[lead] = {
            "rmse_clim": rmse_clim,
            "rmse_raw": rmse_raw,
            "skill_raw": 1 - rmse_raw / rmse_clim if rmse_clim > 0 else np.nan,
            "../results/diagnostics/corr_raw": _corr(yev, raw_pred),
            "rmse_anom": rmse_anom,
            "skill_anom": 1 - rmse_anom / rmse_clim_a if rmse_clim_a > 0 else np.nan,
            "corr_anom": _corr(yev_a, anom_pred),
            "n": len(yev),
        }
    return results


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_ecmv   # noqa: F821  <- replace
    imd_ds = ds_imd      # noqa: F821  <- replace

    t0 = time.perf_counter()

    with stage("Reducing to spatial-mean series (only expensive step)"):
        feat, imd = reduce_to_series(ecmwf_ds, imd_ds)
        mb = sum(v.nbytes for v in feat.data_vars.values()) / 1e6
        print(f"    {dict(feat.sizes)}  ({mb:.1f} MB)")

    with stage("Checking for accumulated fluxes"):
        feat = ensure_valid_time(feat)
        feat = deaccumulate(feat, detect_accumulated(feat))

    with stage("Attaching IMD target"):
        combined = attach_target(feat, imd)
        feature_vars = [v for v in combined.data_vars if v != TARGET_NAME]

    with stage("Splitting by init year"):
        yrs = sorted(set(combined["time"].dt.year.values.tolist()))
        test_years, val_years = set(yrs[-3:]), set(yrs[-6:-3])
        train_ds, val_ds, test_ds = split_by_year(combined, test_years, val_years)
        print(f"    train {train_ds.sizes['time']} | val {val_ds.sizes['time']} "
              f"| test {test_ds.sizes['time']} inits")
        print(f"    features ({len(feature_vars)}): {feature_vars}")

    with stage("Building climatologies (train inits only)"):
        clim_obs = build_obs_climatology(train_ds)
        clim_mod = build_model_climatology(train_ds, feature_vars)

    with stage("Running all three arms"):
        scores = run_baselines(train_ds, val_ds, feature_vars, clim_obs, clim_mod)

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s\n")
    print("Validation skill by lead. Reference = DOY climatology (not a flat")
    print("annual mean). skill > 0 means it beats knowing what date it is.\n")
    print(f"{'lead':>4} {'rmse_clim':>9} {'rmse_raw':>9} {'skl_raw':>8} "
          f"{'rmse_anom':>9} {'skl_anom':>8} {'corr_anom':>9} {'n':>5}")
    for lead, m in sorted(scores.items()):
        print(f"{lead:>4} {m['rmse_clim']:>9.3f} {m['rmse_raw']:>9.3f} "
              f"{m['skill_raw']:>8.3f} {m['rmse_anom']:>9.3f} "
              f"{m['skill_anom']:>8.3f} {m['corr_anom']:>9.3f} {m['n']:>5}")

[ ] Reducing to spatial-mean series (only expensive step) ...
    mask: 4964/17415 cells valid (28.5%)
    computing spatial means (single pass, threaded)...
[########################################] | 100% Completed | 374.65 s
    {'step': 43, 'time': 3720}  (26.9 MB)
[x] Reducing to spatial-mean series (only expensive step)  (376.9s)
[ ] Checking for accumulated fluxes ...
    step is int64 (max 42) -> reading as units='D'. If wrong, set STEP_UNITS explicitly.
    no accumulated vars detected; leaving steps as-is
[x] Checking for accumulated fluxes  (0.0s)
[ ] Attaching IMD target ...
[x] Attaching IMD target  (0.0s)
[ ] Splitting by init year ...
    train 2604 | val 558 | test 558 inits
    features (21): ['10m_u_component_of_wind', '10m_v_component_of_wind', '2m_temperature', 'mean_sea_level_pressure', 'specific_humidity_1000', 'specific_humidity_200', 'specific_humidity_500', 'specific_humidity_850', 'temperature_1000', 'temperature_200', 'temperature_500', 'temperature_850', 't

/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_47627/503083680.py:342: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(X, axis=0) if len(X) else np.zeros(X.shape[1])
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_47627/503083680.py:342: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(X, axis=0) if len(X) else np.zeros(X.shape[1])


[x] Running all three arms  (0.3s)

Total: 380.0s

Validation skill by lead. Reference = DOY climatology (not a flat
annual mean). skill > 0 means it beats knowing what date it is.

lead rmse_clim  rmse_raw  skl_raw rmse_anom skl_anom corr_anom     n
   0     2.008     1.661    0.173     1.475    0.265     0.667   558
   1     2.017     0.710    0.648     0.706    0.650     0.936   558
   2     2.012     0.838    0.583     0.832    0.587     0.913   558
   3     2.024     0.925    0.543     0.925    0.543     0.886   558
   4     2.013     1.081    0.463     1.073    0.467     0.844   558
   5     2.044     1.230    0.398     1.233    0.396     0.789   558
   6     2.007     1.245    0.380     1.234    0.385     0.784   558
   7     2.082     1.390    0.332     1.388    0.333     0.734   558
   8     2.011     1.423    0.292     1.405    0.301     0.706   558
   9     2.084     1.546    0.258     1.542    0.260     0.660   558
  10     1.977     1.590    0.196     1.570    0.206     0.

In [15]:
feat.to_netcdf("../data/processed/feat_series.nc"); imd.to_netcdf("../data/processed/imd_series.nc")

In [8]:
"""
GRIDDED per-cell S2S baseline: one linear regression per (valid cell, lead),
in anomaly space, scored against a per-cell DOY climatology. Produces skill
MAPS, not a single all-India number.

Companion to the spatial-mean version. Same three arms (clim / raw / anom),
same references, but nothing is averaged away.


WHY THIS FITS ON A LAPTOP
-------------------------
Full 4-D would be 2604 x 43 x 4964 x 21 x 4B = 47 GB. We never build it.

  1. MASK FIRST. Only 4964 of 17415 cells (28.5%) are inside IMD's domain.
     We interpolate DIRECTLY to those 4964 points -- the 129x135 rectangle
     is never materialised at all.
  2. BATCH BY LEAD. One lead of training data is 2604 x 4964 x 21 x 4B
     = 1.1 GB. That fits; 43 of them at once does not.
  3. LOAD COARSE ONCE. The ECMWF subset over India at 1.5deg is small.
     Load it to RAM once, then interpolate per lead from memory. If instead
     you re-read zarr per lead and `step` is a single chunk, you re-read the
     whole archive 43 times (~4.5 hours vs ~6 minutes).

clim_mod gridded would be 21 x 366 x 43 x 4964 x 4B = 6.6 GB, so it is built
INSIDE the lead loop (~150 MB/lead), never all at once.


WHY BATCHED NORMAL EQUATIONS, NOT sklearn
-----------------------------------------
Honestly: not for speed. Benchmarked at 4964 cells / 2604 samples / 21
features, the batched solve is 6.7s vs 6.8s for an sklearn loop -- identical.
~5 minutes either way for 43 leads.

The reason to use it is memory control. Written naively, the batched form
makes four full 1.1 GB copies before reaching BLAS and gets OOM-killed on a
16 GB machine. Chunked over cells it peaks around 3.4 GB. If you prefer an
sklearn loop, that is a perfectly reasonable choice -- just keep the
per-cell standardisation.
"""

import time
from contextlib import contextmanager

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
CLIM_WINDOW_DAYS = 3
STEP_UNITS = "auto"
RIDGE = 1e-4            # on the standardised normal equations
CELL_CHUNK = 256        # cells per solve block -- controls peak RAM, not speed
DTYPE = np.float32

# Subsetting knobs -- these control whether this fits in RAM.
LEADS = None            # e.g. range(14, 43) for your target window; None = all
MONTHS = None           # e.g. (6, 7, 8, 9) for JJAS only; None = all year
COARSE_PAD = 2.0        # degrees of ECMWF kept beyond IMD's box (for interp edges)

OUT_SKILL_MAPS = "../results/models/skill_maps.nc"


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- step / time plumbing ----------------

def normalize_step(ds, units=STEP_UNITS, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    if not np.issubdtype(v.dtype, np.number):
        raise TypeError(f"'step' has dtype {v.dtype}; expected timedelta64 or numeric.")
    u = units
    if u == "auto":
        mx = int(np.nanmax(v))
        u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
        if verbose:
            print(f"    step is {v.dtype} (max {mx}) -> reading as units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def step_days(ds):
    return (ds["step"].values / np.timedelta64(1, "D")).astype(int)


# ---------------- cells ----------------

def build_cells(imd_ds, var=IMD_TARGET_VAR):
    """Flatten IMD to a 1-D list of VALID cells only.

    Everything downstream works on (time, cell) instead of (time, lat, lon),
    which drops 71.5% of the grid before any arithmetic happens. The lat/lon
    of each cell are kept so we can interpolate ECMWF straight to these
    points and so we can unstack skill back into a map at the end.
    """
    da = imd_ds[var]
    mask2d = da.notnull().any(dim="time").compute()
    stacked = mask2d.stack(cell=("lat", "lon"))
    sel = stacked.values
    cells = stacked[sel]
    n, tot = int(sel.sum()), int(sel.size)
    print(f"    {n}/{tot} valid cells ({100 * n / tot:.1f}%) -- "
          f"interpolating straight to these, skipping the rest")
    return cells, mask2d


def cell_points(cells):
    lat = xr.DataArray(cells["lat"].values, dims="cell")
    lon = xr.DataArray(cells["lon"].values, dims="cell")
    return lat, lon


# ---------------- loading ----------------

def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, leads=LEADS, pad=COARSE_PAD):
    """ECMWF over India's box, still at native coarse res, fully in RAM.

    This is the ONE disk pass. Keeping it coarse is what makes it small:
    the 0.25 expansion happens per lead, in memory, and is discarded.
    """
    ecmwf_ds = ensure_valid_time(ecmwf_ds)

    # lat may be descending (ECMWF convention); slice() silently returns
    # empty on a descending index, so normalise first.
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)

    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad),
                       lon=slice(lon0 - pad, lon1 + pad))

    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    if leads is not None:
        sub = sub.isel(step=list(leads))

    nbytes = np.prod([sub.sizes[d] for d in ("time", "step", "lat", "lon")]) \
        * len(sub.data_vars) * 4
    print(f"    coarse subset {dict(sub.sizes)} x {len(sub.data_vars)} vars "
          f"-> {nbytes / 1e9:.2f} GB in RAM")
    if nbytes > 8e9:
        print("    WARNING: >8 GB. Set MONTHS=(6,7,8,9) and/or LEADS=range(14,43).")

    with ProgressBar():
        sub = sub.astype(DTYPE).compute()
    return sub


def load_target(imd_ds, cells, var=IMD_TARGET_VAR):
    """IMD as (time, cell), valid cells only. ~KBs per day."""
    da = imd_ds[var].stack(cell=("lat", "lon"))
    da = da.sel(cell=cells["cell"])
    with ProgressBar():
        da = da.astype(DTYPE).compute()
    return da.assign_coords(time=da["time"].dt.floor("D"))


def lead_slice(coarse, j, lat_pts, lon_pts, feature_vars):
    """Interpolate ONE lead's coarse fields to the 4964 valid points.

    Pointwise interp (lat/lon as DataArrays sharing dim 'cell') means the
    129x135 rectangle is never allocated -- we only ever touch valid cells.
    Returns (n_time, n_cell, n_feat) float32, ~1.1 GB for train.
    """
    s = coarse.isel(step=j)
    fine = s[feature_vars].interp(lat=lat_pts, lon=lon_pts)
    return np.stack([fine[v].values for v in feature_vars], axis=-1).astype(DTYPE)


# ---------------- climatology ----------------

def _doy_matrix(doys, window, n_doy=366):
    """(n_doy, n_samples) circular-window membership matrix."""
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """values (n, c) -> (n_doy, c) DOY climatology per cell, NaN-aware.

    The NaN handling is a matmul trick: counts = M @ isfinite gives the
    per-(doy, cell) valid sample count without ever building an (n_doy, n, c)
    boolean array (which would be ~5e9 elements here).
    """
    M = _doy_matrix(doys, window)
    finite = np.isfinite(values).astype(DTYPE)
    counts = M @ finite
    sums = M @ np.nan_to_num(values).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.astype(DTYPE), counts


# ---------------- batched per-cell regression ----------------

def fit_predict_cells(Xtr, ytr, Xev, ridge=RIDGE, chunk=CELL_CHUNK):
    """One regression per cell, chunked over cells.

    Xtr (n, c, f), ytr (n, c), Xev (m, c, f) -> pred (m, c)

    CHUNKING IS ABOUT MEMORY, NOT SPEED. Benchmarked at 4964 cells / 2604
    samples / 21 features, this is 6.7s vs 6.8s for an sklearn loop -- i.e.
    the same. What it buys is peak RSS: the unchunked version makes four
    full 1.1 GB copies (subtract, divide, nan_to_num, transpose) before it
    reaches BLAS and gets OOM-killed on a 16 GB machine. Chunked peaks ~3.4 GB.

    Standardise per cell before the solve. This is load-bearing, not hygiene:
    msl ~1e5 Pa and specific humidity ~1e-2 kg/kg differ by seven orders of
    magnitude, and float32 normal equations on that are singular in practice.
    After centering, no intercept column is needed.
    """
    n, c, f = Xtr.shape
    m = Xev.shape[0]
    out = np.empty((m, c), DTYPE)

    for a in range(0, c, chunk):
        b = min(a + chunk, c)
        xt, xe, yt = Xtr[:, a:b], Xev[:, a:b], ytr[:, a:b]

        mu = np.nanmean(xt, axis=0)
        sd = np.nanstd(xt, axis=0)
        sd = np.where(sd < 1e-12, 1.0, sd)      # dead predictors -> no-op
        ymu = np.nanmean(yt, axis=0)

        xs = np.nan_to_num((xt - mu) / sd)
        ys = np.nan_to_num(yt - ymu)

        XtX = np.einsum("ncf,ncg->cfg", xs, xs, optimize=True)
        Xty = np.einsum("ncf,nc->cf", xs, ys, optimize=True)
        XtX += (ridge * n) * np.eye(f, dtype=XtX.dtype)

        beta = np.linalg.solve(XtX, Xty[:, :, None])[:, :, 0]
        out[:, a:b] = np.einsum(
            "mcf,cf->mc", np.nan_to_num((xe - mu) / sd), beta) + ymu

    return out


def _rmse_cells(y, p):
    """Per-cell RMSE ignoring NaN. y, p: (n, c) -> (c,)"""
    e2 = (y - p) ** 2
    with np.errstate(invalid="ignore"):
        return np.sqrt(np.nanmean(e2, axis=0))


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_ecmv   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    t0 = time.perf_counter()

    with stage("Building valid-cell index"):
        cells, mask2d = build_cells(imd_ds)
        lat_pts, lon_pts = cell_points(cells)

    with stage("Loading coarse ECMWF into RAM (the only disk pass)"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)
        leads = step_days(coarse)

    with stage("Loading IMD target"):
        imd_cells = load_target(imd_ds, cells)

    with stage("Splitting by init year"):
        yrs = sorted(set(coarse["time"].dt.year.values.tolist()))
        test_years, val_years = set(yrs[-3:]), set(yrs[-6:-3])
        y_of = coarse["time"].dt.year
        tr_i = ~(y_of.isin(list(test_years)) | y_of.isin(list(val_years))).values
        va_i = y_of.isin(list(val_years)).values
        print(f"    train {tr_i.sum()} | val {va_i.sum()} inits, "
              f"{len(feature_vars)} features, {len(leads)} leads")

    n_cell = len(cells["cell"])
    rows = []
    maps = {"skill_anom": [], "skill_raw": [], "rmse_clim": []}

    with stage(f"Fitting {n_cell} cells x {len(leads)} leads"):
        for j, lead in enumerate(leads):
            Xall = lead_slice(coarse, j, lat_pts, lon_pts, feature_vars)

            vt = coarse["valid_time"].isel(step=j).dt.floor("D").values
            yall = imd_cells.reindex(time=vt).values                 # (n, c)
            doy = xr.DataArray(vt, dims="t").dt.dayofyear.values

            Xtr, ytr, dtr = Xall[tr_i], yall[tr_i], doy[tr_i]
            Xev, yev, dev = Xall[va_i], yall[va_i], doy[va_i]

            # --- climatologies, train only, this lead only (~150 MB) ---
            clim_o, _ = _clim_grid(ytr, dtr, CLIM_WINDOW_DAYS)       # (366, c)
            clim_m = np.empty((366, n_cell, len(feature_vars)), DTYPE)
            for k in range(len(feature_vars)):
                clim_m[:, :, k], _ = _clim_grid(Xtr[:, :, k], dtr, CLIM_WINDOW_DAYS)

            # --- arm 1: per-cell DOY climatology (the reference) ---
            p_clim = clim_o[dev - 1]
            rmse_clim = _rmse_cells(yev, p_clim)

            # --- arm 2: raw ---
            p_raw = fit_predict_cells(Xtr, ytr, Xev)
            rmse_raw = _rmse_cells(yev, p_raw)

            # --- arm 3: anomaly (each side minus its OWN climatology;
            #     model clim is lead-dependent, obs clim is not) ---
            Xtr_a = Xtr - clim_m[dtr - 1]
            Xev_a = Xev - clim_m[dev - 1]
            ytr_a = ytr - clim_o[dtr - 1]
            yev_a = yev - clim_o[dev - 1]
            p_anom = fit_predict_cells(Xtr_a, ytr_a, Xev_a)
            rmse_anom = _rmse_cells(yev_a, p_anom)
            rmse_clim_a = _rmse_cells(yev_a, np.zeros_like(yev_a))

            with np.errstate(invalid="ignore", divide="ignore"):
                sk_raw = 1 - rmse_raw / rmse_clim
                sk_anom = 1 - rmse_anom / rmse_clim_a

            maps["skill_anom"].append(sk_anom)
            maps["skill_raw"].append(sk_raw)
            maps["rmse_clim"].append(rmse_clim)
            rows.append((int(lead), np.nanmean(rmse_clim), np.nanmean(rmse_raw),
                         np.nanmean(sk_raw), np.nanmean(rmse_anom),
                         np.nanmean(sk_anom), float(np.nanmean(sk_anom > 0))))

            del Xall, Xtr, Xev, clim_m
            print(f"    lead {int(lead):>2}: mean skill_anom {np.nanmean(sk_anom):+.3f} "
                  f"| {100 * np.nanmean(sk_anom > 0):.0f}% of cells positive", flush=True)

    with stage("Writing skill maps"):
        out = xr.Dataset(
            {k: (("lead", "cell"), np.stack(v)) for k, v in maps.items()},
            coords={"lead": leads, "cell": cells["cell"]},
        ).unstack("cell")
        out.to_netcdf(OUT_SKILL_MAPS)
        print(f"    -> {OUT_SKILL_MAPS}  {dict(out.sizes)}")

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s\n")
    print("Domain-MEAN skill by lead (maps are in the netcdf; means hide "
          "regional structure -- plot them).\n")
    print(f"{'lead':>4} {'rmse_clim':>9} {'rmse_raw':>9} {'skl_raw':>8} "
          f"{'rmse_anom':>9} {'skl_anom':>8} {'%cells+':>8}")
    for r in rows:
        print(f"{r[0]:>4} {r[1]:>9.3f} {r[2]:>9.3f} {r[3]:>8.3f} "
              f"{r[4]:>9.3f} {r[5]:>8.3f} {100 * r[6]:>7.0f}%")

[ ] Building valid-cell index ...
    4964/17415 valid cells (28.5%) -- interpolating straight to these, skipping the rest
[x] Building valid-cell index  (0.9s)
[ ] Loading coarse ECMWF into RAM (the only disk pass) ...
    step is int64 (max 42) -> reading as units='D'
    coarse subset {'time': 3720, 'step': 43, 'lat': 33, 'lon': 35} x 21 vars -> 15.52 GB in RAM
[###########                             ] | 27% Completed | 2.82 s ms


KeyboardInterrupt: 

In [10]:
"""
Diagnostic 1: per-lead box plots of variable-vs-rain correlation, distribution
              taken across the 4964 valid IMD cells (one box per predictor).
Diagnostic 2: the same, stratified into four regional zones, plus the
              inter-zone rain correlation matrix.


READ THIS FIRST: CORRELATIONS ARE COMPUTED ON ANOMALIES
-------------------------------------------------------
Correlating RAW predictor against RAW rain gives r ~ 0.8 for almost every
variable at almost every lead, because both carry the monsoon annual cycle.
That is the same artefact that made a raw linear regression look like it had
corr 0.84 skill at day 42 when its true skill was ~0.01. Every box plot would
look magnificent and mean nothing.

So: each side is de-climatologised first. Predictors lose a LEAD-DEPENDENT
model climatology (drift grows with lead); rain loses a DOY-only observed
climatology. Climatology is fit on TRAIN inits only, so these plots stay
honest if you later reuse them next to skill numbers.

Set ANOMALY = False only if you specifically want to see the artefact.


ON THE FOUR ZONES -- THE HONEST VERSION
---------------------------------------
IMD's four homogeneous regions (NWI, CI, NEI, SPIN) are NOT lat/lon boxes.
Parthasarathy et al. (1995) define them as GROUPS OF METEOROLOGICAL
SUBDIVISIONS -- polygons following state/district boundaries. Zheng et al.
(2016, JGR, 10.1002/2016JD025135), doing this exact projection, states that
gridded rainfall "can be projected only approximately onto the four IMD
regions due primarily to the challenge of exactly delineating the complex
boundary between meteorological subdivisions by grid box."

Therefore ZONES_APPROX below is an APPROXIMATION, not a literature standard.
Do not report it as "the IMD homogeneous regions". Either:
  (a) call it "approximate zones following Zheng et al. (2016)" and cite that
      the projection is approximate, or
  (b) set ZONE_SHAPEFILE to IMD's 36-subdivision shapefile and group them --
      the correct, citable route. assign_zones() supports both.
"""

import os
import time
from contextlib import contextmanager

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
ANOMALY = True                  # False reproduces the seasonal-cycle artefact
CLIM_WINDOW_DAYS = 0
DTYPE = np.float32
COARSE_PAD = 2.0
LEADS = None                    # e.g. range(14, 43); None = all
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS
OUTDIR = "../results/diagnostics/corr_diagnostics_3"

LAT_CUT, LON_CUT = 23.5, 82.5

ZONES_APPROX = [
    ("NW", lambda la, lo: (la >= LAT_CUT) & (lo <  LON_CUT)),
    ("NE", lambda la, lo: (la >= LAT_CUT) & (lo >= LON_CUT)),
    ("SW", lambda la, lo: (la <  LAT_CUT) & (lo <  LON_CUT)),
    ("SE", lambda la, lo: (la <  LAT_CUT) & (lo >= LON_CUT)),
]


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- shared plumbing (same as the baseline scripts) ----------------

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def build_cells(imd_ds, var=IMD_TARGET_VAR):
    mask2d = imd_ds[var].notnull().any(dim="time").compute()
    stacked = mask2d.stack(cell=("lat", "lon"))
    cells = stacked[stacked.values]
    print(f"    {len(cells['cell'])} valid cells")
    return cells


def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, leads=LEADS, pad=COARSE_PAD):
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)
    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad), lon=slice(lon0 - pad, lon1 + pad))
    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    if leads is not None:
        sub = sub.isel(step=list(leads))
    with ProgressBar():
        return sub.astype(DTYPE).compute()


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, c) -> (366, c), NaN-aware via matmul counts."""
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(values).astype(DTYPE)
    sums = M @ np.nan_to_num(values).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan).astype(DTYPE)


# ---------------- zones ----------------

def assign_zones(cells, shapefile=ZONE_SHAPEFILE):
    """Return (n_cell,) array of zone labels.

    shapefile route = correct (IMD's actual subdivision polygons).
    box route = approximation; label it as such in any figure caption.
    """
    lat = cells["lat"].values
    lon = cells["lon"].values

    if shapefile is not None:
        import geopandas as gpd
        from shapely.geometry import Point
        gdf = gpd.read_file(shapefile)
        pts = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(lon, lat)],
                               crs=gdf.crs)
        joined = gpd.sjoin(pts, gdf, how="left", predicate="within")
        # NOTE: you must map subdivision names -> NWI/CI/NEI/SPIN per
        # Parthasarathy et al. (1995) Table 1. Column name varies by file.
        return joined.iloc[:, -1].values

    z = np.empty(len(lat), dtype=object)
    unassigned = np.ones(len(lat), bool)
    for name, rule in ZONES_APPROX:
        m = rule(lat, lon) & unassigned
        z[m] = name
        unassigned &= ~m
    return z


# ---------------- correlation ----------------

def corr_cells(X, y):
    """X (n, c, f), y (n, c) -> (c, f) Pearson r per cell per variable."""
    Xc = X - np.nanmean(X, axis=0)
    yc = y - np.nanmean(y, axis=0)
    num = np.nansum(Xc * yc[:, :, None], axis=0)
    den = np.sqrt(np.nansum(Xc ** 2, axis=0) * np.nansum(yc ** 2, axis=0)[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def corr_1d(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    a, b = x[m] - x[m].mean(), y[m] - y[m].mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ---------------- plotting ----------------

def boxplot_lead(r, feature_vars, lead, outdir, anomaly):
    """r: (c, f) per-cell correlations. One box per variable."""
    order = np.argsort(-np.nanmedian(np.abs(r), axis=0))
    data = [r[:, k][np.isfinite(r[:, k])] for k in order]
    labels = [feature_vars[k] for k in order]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, whis=(5, 95))
    for p in bp["boxes"]:
        p.set_facecolor("#4C78A8")
        p.set_alpha(0.65)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Pearson r vs rain (per cell)")
    ax.set_ylim(-1, 1)
    kind = "anomaly" if anomaly else "RAW (seasonal cycle included!)"
    ax.set_title(f"Lead {lead} d  --  {kind} space, "
                 f"distribution across {len(data[0])} cells, sorted by |median r|")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"3_day_corr_box_lead{lead:02d}.png"), dpi=110)
    plt.close(fig)


def summary_plot(med, feature_vars, leads, outdir):
    """median |r| vs lead, one line per variable. The plot that answers
    'which predictors survive to week 3+', which 43 separate boxes cannot."""
    fig, ax = plt.subplots(figsize=(11, 6))
    final = np.nanmean(med[-5:], axis=0)
    for k in np.argsort(-final):
        ax.plot(leads, med[:, k], lw=1.4, label=feature_vars[k])
    ax.set_xlabel("lead (days)")
    ax.set_ylabel("median |r| across cells")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    ax.set_title("Predictor-rain correlation decay by lead (anomaly space)")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "3_day_summary_corr_vs_lead.png"), dpi=130)
    plt.close(fig)


def zone_heatmaps(zr, zones, feature_vars, leads, outdir):
    """zr: (n_lead, n_zone, n_feat). One heatmap per zone."""
    fig, axes = plt.subplots(1, len(zones), figsize=(5.5 * len(zones), 7), sharey=True)
    for zi, (ax, zname) in enumerate(zip(np.atleast_1d(axes), zones)):
        im = ax.imshow(zr[:, zi, :].T, aspect="auto", cmap="RdBu_r",
                       vmin=-0.8, vmax=0.8,
                       extent=[leads[0], leads[-1], len(feature_vars) - 0.5, -0.5])
        ax.set_yticks(range(len(feature_vars)))
        ax.set_yticklabels(feature_vars, fontsize=7)
        ax.set_xlabel("lead (days)")
        ax.set_title(zname)
    fig.colorbar(im, ax=axes, label="r (zone-mean anomaly vs zone-mean rain anomaly)")
    fig.savefig(os.path.join(outdir, "3_day_zone_corr_heatmap.png"), dpi=130,
                bbox_inches="tight")
    plt.close(fig)


def zone_map(cells, zlab, zones, outdir):
    """Sanity check: LOOK AT THIS. Box-based zones will have straight edges
    that cut through states. If that offends you, use the shapefile."""
    da = xr.DataArray(np.array([zones.index(z) for z in zlab], float),
                      coords={"cell": cells["cell"]}, dims="cell").unstack("cell")
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.pcolormesh(da["lon"], da["lat"], da.values, cmap="tab10", vmin=0, vmax=9)
    cb = fig.colorbar(im, ax=ax, ticks=range(len(zones)))
    cb.ax.set_yticklabels(zones)
    ax.set_title("Zone assignment (APPROXIMATE -- not IMD's polygons)")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "3_day_zone_map.png"), dpi=130)
    plt.close(fig)


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_new   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    os.makedirs(OUTDIR, exist_ok=True)
    t0 = time.perf_counter()

    with stage("Setup"):
        cells = build_cells(imd_ds)
        lat_pts = xr.DataArray(cells["lat"].values, dims="cell")
        lon_pts = xr.DataArray(cells["lon"].values, dims="cell")
        zlab = assign_zones(cells)
        zones = [z for z, _ in ZONES_APPROX] if ZONE_SHAPEFILE is None \
            else sorted(set(zlab))
        for z in zones:
            print(f"    zone {z}: {(zlab == z).sum()} cells")

    with stage("Loading coarse ECMWF"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)
        leads = (coarse["step"].values / np.timedelta64(1, "D")).astype(int)

    with stage("Loading IMD"):
        imd_cells = imd_ds[IMD_TARGET_VAR].stack(cell=("lat", "lon")).sel(
            cell=cells["cell"]).astype(DTYPE)
        with ProgressBar():
            imd_cells = imd_cells.compute()
        imd_cells = imd_cells.assign_coords(time=imd_cells["time"].dt.floor("D"))

    yrs = sorted(set(coarse["time"].dt.year.values.tolist()))
    tr_i = ~coarse["time"].dt.year.isin(yrs[-6:]).values   # climatology on train only

    n_z, n_f = len(zones), len(feature_vars)
    med = np.full((len(leads), n_f), np.nan)
    zr = np.full((len(leads), n_z, n_f), np.nan)
    zone_rain = np.full((len(leads), n_z, coarse.sizes["time"]), np.nan)

    with stage(f"Correlations: {len(leads)} leads x {n_f} vars"):
        for j, lead in enumerate(leads):
            s = coarse.isel(step=j)
            X = np.stack([s[v].interp(lat=lat_pts, lon=lon_pts).values
                          for v in feature_vars], axis=-1).astype(DTYPE)
            vt = s["valid_time"].dt.floor("D").values
            y = imd_cells.reindex(time=vt).values
            doy = xr.DataArray(vt, dims="t").dt.dayofyear.values

            if ANOMALY:
                # obs clim: DOY only. model clim: DOY x THIS lead (drift).
                co = _clim_grid(y[tr_i], doy[tr_i], CLIM_WINDOW_DAYS)
                y = y - co[doy - 1]
                for k in range(n_f):
                    cm = _clim_grid(X[tr_i, :, k], doy[tr_i], CLIM_WINDOW_DAYS)
                    X[:, :, k] -= cm[doy - 1]

            r = corr_cells(X, y)                      # (c, f)
            med[j] = np.nanmedian(np.abs(r), axis=0)
            boxplot_lead(r, feature_vars, int(lead), OUTDIR, ANOMALY)

            for zi, z in enumerate(zones):
                m = (zlab == z)
                yz = np.nanmean(y[:, m], axis=1)
                zone_rain[j, zi] = yz
                for k in range(n_f):
                    zr[j, zi, k] = corr_1d(np.nanmean(X[:, m, k], axis=1), yz)

            del X, y
            print(f"    lead {int(lead):>2} done", flush=True)

    with stage("Summary figures"):
        summary_plot(med, feature_vars, leads, OUTDIR)
        zone_heatmaps(zr, zones, feature_vars, leads, OUTDIR)
        zone_map(cells, zlab, zones, OUTDIR)

        # inter-zone rain correlation, per lead
        print(f"\n    Inter-zone rain-anomaly correlation (lead {leads[-1]}):")
        print("         " + "".join(f"{z:>7}" for z in zones))
        M = np.eye(n_z)
        for a in range(n_z):
            row = [corr_1d(zone_rain[-1, a], zone_rain[-1, b]) for b in range(n_z)]
            M[a] = row
            print(f"    {zones[a]:>5}" + "".join(f"{v:>7.2f}" for v in row))
        np.save(os.path.join(OUTDIR, "interzone_corr.npy"), M)

        xr.Dataset(
            {"median_abs_r": (("lead", "var"), med),
             "zone_r": (("lead", "zone", "var"), zr)},
            coords={"lead": leads, "var": feature_vars, "zone": zones},
        ).to_netcdf(os.path.join(OUTDIR, "correlations.nc"))

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s")
    print(f"{len(leads)} box plots + summary + zone heatmap -> {OUTDIR}/")
    print("\nLook at summary_corr_vs_lead.png first -- 43 box plots are hard to")
    print("read as a set; the summary is what answers 'which predictors survive'.")

NameError: name 'ZONE_SHAPEFILE' is not defined

In [1]:
"""
WINDOWED correlation diagnostics for S2S.

Both predictors AND target are averaged over the same forecast window before
correlating. That is the whole point: independent day-to-day noise cancels in
each average, and only variability coherent across the window survives on both
sides to correlate. Windowing the target alone would mostly re-measure the
daily relationship.

Windows follow the S2S field standard (ECMWF operational products; Horat &
Lerch 2024 MWR; the ECMWF S2S AI Challenge), not arbitrary choices:

    week2      days  8-14    reference point
    week3-4    days 15-28    the headline S2S target
    week5-6    days 29-42    the hard one
    d14-42     days 14-42    your stated target window
    week3      days 15-21    finer resolution, if you want it
    week4      days 22-28
    week5      days 29-35
    week6      days 36-42

For each window: aggregate predictors over the window's leads, aggregate IMD
rain over the SAME valid dates, de-climatologise both, correlate per cell.

Anomalies are mandatory here. Windowed RAW values correlate at r ~ 0.9 for
almost everything -- window-averaging strengthens the seasonal cycle (it is
coherent across the window by construction) while cancelling weather noise.
Raw windowed correlation is the artefact at its most flattering.

Climatology is fit on TRAIN inits only. Same leakage rule as the baselines.
"""

import os
import time
from contextlib import contextmanager

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
DTYPE = np.float32
COARSE_PAD = 2.0
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits only
CLIM_WINDOW_DAYS = 7
OUTDIR = "../results/diagnostics/corr_windowed"

# (name, first_lead_day, last_lead_day) inclusive
WINDOWS = [
    ("week2",   8, 14),
    ("week3-4", 15, 28),
    ("week5-6", 29, 42),
    ("d14-42",  14, 42),
    ("week3",   15, 21),
    ("week4",   22, 28),
    ("week5",   29, 35),
    ("week6",   36, 42),
]

WINDOW_PREDICTORS = True        # False -> window target only (the weaker variant)
COMPARE_DAILY_LEAD = 22         # single lead to plot alongside, for contrast

ZONE_SHAPEFILE = None
LAT_CUT, LON_CUT = 23.5, 82.5   # Tropic of Cancer / India's standard meridian
ZONES = [
    ("NW", lambda la, lo: (la >= LAT_CUT) & (lo < LON_CUT)),
    ("NE", lambda la, lo: (la >= LAT_CUT) & (lo >= LON_CUT)),
    ("SW", lambda la, lo: (la < LAT_CUT) & (lo < LON_CUT)),
    ("SE", lambda la, lo: (la < LAT_CUT) & (lo >= LON_CUT)),
]


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- plumbing ----------------

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def build_cells(imd_ds, var=IMD_TARGET_VAR):
    mask2d = imd_ds[var].notnull().any(dim="time").compute()
    st = mask2d.stack(cell=("lat", "lon"))
    cells = st[st.values]
    print(f"    {len(cells['cell'])}/{mask2d.size} valid cells")
    return cells


def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, pad=COARSE_PAD):
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)
    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad),
                       lon=slice(lon0 - pad, lon1 + pad))
    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    print(f"    {dict(sub.sizes)} x {len(sub.data_vars)} vars")
    with ProgressBar():
        return sub.astype(DTYPE).compute()


def assign_zones(cells):
    lat, lon = cells["lat"].values, cells["lon"].values
    z = np.empty(len(lat), dtype=object)
    un = np.ones(len(lat), bool)
    for name, rule in ZONES:
        m = rule(lat, lon) & un
        z[m] = name
        un &= ~m
    return z


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, c) -> (366, c). NaN-aware without ever building (366, n, c)."""
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(values).astype(DTYPE)
    sums = M @ np.nan_to_num(values).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan).astype(DTYPE)


def corr_cells(X, y):
    """X (n, c, f), y (n, c) -> (c, f) Pearson r per cell per variable."""
    Xc = X - np.nanmean(X, axis=0)
    yc = y - np.nanmean(y, axis=0)
    num = np.nansum(Xc * yc[:, :, None], axis=0)
    den = np.sqrt(np.nansum(Xc ** 2, axis=0) * np.nansum(yc ** 2, axis=0)[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def corr_1d(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    a, b = x[m] - x[m].mean(), y[m] - y[m].mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ---------------- windowed aggregation ----------------

def window_slice(coarse, imd_cells, lo, hi, lat_pts, lon_pts, feature_vars,
                 window_predictors=WINDOW_PREDICTORS):
    """Aggregate predictors and target over lead days [lo, hi] inclusive.

    Predictors: mean over the window's leads, then interp to valid cells.
    Averaging on the COARSE grid first (before interp) is exact and ~6x
    cheaper -- both operations are linear, so they commute.

    Target: mean of IMD rain over the SAME valid dates the window spans, i.e.
    init+lo .. init+hi. This is the bit that must line up; getting the target
    window from anything other than the forecast's own valid dates silently
    correlates against the wrong fortnight.
    """
    leads = (coarse["step"].values / np.timedelta64(1, "D")).astype(int)
    sel = np.where((leads >= lo) & (leads <= hi))[0]
    if len(sel) == 0:
        raise ValueError(f"no leads in [{lo}, {hi}]; have {leads.min()}..{leads.max()}")

    if window_predictors:
        agg = coarse.isel(step=sel).mean(dim="step")
    else:
        mid = sel[len(sel) // 2]
        agg = coarse.isel(step=mid)

    X = np.stack([agg[v].interp(lat=lat_pts, lon=lon_pts).values
                  for v in feature_vars], axis=-1).astype(DTYPE)

    # target: average IMD over each init's own window of valid dates
    vt = coarse["valid_time"].isel(step=sel).dt.floor("D").values   # (n_init, n_lead)
    flat = vt.ravel()
    y_all = imd_cells.reindex(time=flat).values                     # (n_init*n_lead, c)
    y = np.nanmean(y_all.reshape(vt.shape + (-1,)), axis=1)         # (n_init, c)

    # label the window by its centre valid date, for climatology lookup
    centre = coarse["time"].values + np.timedelta64((lo + hi) // 2, "D")
    doy = xr.DataArray(centre, dims="t").dt.dayofyear.values
    return X, y, doy, len(sel)


# ---------------- plots ----------------

def boxplot_window(r, feature_vars, wname, lo, hi, n_leads, outdir):
    order = np.argsort(-np.nanmedian(np.abs(r), axis=0))
    data = [r[:, k][np.isfinite(r[:, k])] for k in order]
    labels = [feature_vars[k] for k in order]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, whis=(5, 95))
    for p in bp["boxes"]:
        p.set_facecolor("#E45756")
        p.set_alpha(0.7)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Pearson r vs windowed rain anomaly (per cell)")
    ax.set_ylim(-1, 1)
    ax.set_title(f"{wname}  (days {lo}-{hi}, {n_leads} leads averaged)  --  "
                 f"anomaly space, {len(data[0])} cells")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"corr_box_{wname}.png"), dpi=110)
    plt.close(fig)


def lift_plot(med_win, med_daily, wnames, feature_vars, outdir, daily_lead):
    """The money plot: windowed vs single-day correlation, same predictors.
    If windowing does nothing, these overlap and the premise was wrong."""
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(feature_vars))
    order = np.argsort(-med_win[wnames.index("week3-4")])
    for wi, w in enumerate(["week2", "week3-4", "week5-6"]):
        ax.plot(x, med_win[wnames.index(w)][order], "o-", lw=1.5, ms=4, label=w)
    ax.plot(x, med_daily[order], "k--", lw=1.2, ms=3,
            label=f"single day (lead {daily_lead})")
    ax.set_xticks(x)
    ax.set_xticklabels([feature_vars[k] for k in order], rotation=90, fontsize=8)
    ax.set_ylabel("median |r| across cells")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title("Does window-averaging lift correlation? "
                 "(both sides windowed, anomaly space)")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "windowed_vs_daily.png"), dpi=130)
    plt.close(fig)


def zone_heatmap(zr, zone_names, feature_vars, wnames, outdir):
    fig, axes = plt.subplots(1, len(zone_names),
                             figsize=(4.2 * len(zone_names), 7), sharey=True)
    axes = np.atleast_1d(axes)
    for zi, (ax, zn) in enumerate(zip(axes, zone_names)):
        im = ax.imshow(zr[:, zi, :].T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(wnames)))
        ax.set_xticklabels(wnames, rotation=90, fontsize=8)
        ax.set_yticks(range(len(feature_vars)))
        ax.set_yticklabels(feature_vars, fontsize=7)
        ax.set_title(zn)
    fig.colorbar(im, ax=axes, label="r (zone-mean anomaly, windowed)")
    fig.savefig(os.path.join(outdir, "zone_corr_windowed.png"), dpi=130,
                bbox_inches="tight")
    plt.close(fig)


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_ecmv   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    os.makedirs(OUTDIR, exist_ok=True)
    t0 = time.perf_counter()

    with stage("Setup"):
        cells = build_cells(imd_ds)
        lat_pts = xr.DataArray(cells["lat"].values, dims="cell")
        lon_pts = xr.DataArray(cells["lon"].values, dims="cell")
        zlab = assign_zones(cells)
        zone_names = [z for z, _ in ZONES]
        for z in zone_names:
            print(f"    zone {z}: {(zlab == z).sum()} cells")

    with stage("Loading coarse ECMWF"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)

    with stage("Loading IMD"):
        imd_cells = imd_ds[IMD_TARGET_VAR].stack(cell=("lat", "lon")).sel(
            cell=cells["cell"]).astype(DTYPE)
        with ProgressBar():
            imd_cells = imd_cells.compute()
        imd_cells = imd_cells.assign_coords(time=imd_cells["time"].dt.floor("D"))

    yrs = sorted(set(coarse["time"].dt.year.values.tolist()))
    tr_i = ~coarse["time"].dt.year.isin(yrs[-6:]).values   # climatology: train only
    print(f"    climatology from {tr_i.sum()} train inits "
          f"(holding out {yrs[-6:]})")

    n_f, n_z = len(feature_vars), len(zone_names)
    wnames = [w[0] for w in WINDOWS]
    med_win = np.full((len(WINDOWS), n_f), np.nan)
    zr = np.full((len(WINDOWS), n_z, n_f), np.nan)
    rows = []

    with stage(f"Windowed correlations: {len(WINDOWS)} windows x {n_f} vars"):
        for wi, (wname, lo, hi) in enumerate(WINDOWS):
            X, y, doy, n_leads = window_slice(
                coarse, imd_cells, lo, hi, lat_pts, lon_pts, feature_vars)

            # de-climatologise both sides. window index plays the role lead
            # played before: the model's drift is a property of the window.
            co = _clim_grid(y[tr_i], doy[tr_i], CLIM_WINDOW_DAYS)
            y = y - co[doy - 1]
            for k in range(n_f):
                cm = _clim_grid(X[tr_i, :, k], doy[tr_i], CLIM_WINDOW_DAYS)
                X[:, :, k] -= cm[doy - 1]

            r = corr_cells(X, y)
            med_win[wi] = np.nanmedian(np.abs(r), axis=0)
            boxplot_window(r, feature_vars, wname, lo, hi, n_leads, OUTDIR)

            for zi, z in enumerate(zone_names):
                m = (zlab == z)
                yz = np.nanmean(y[:, m], axis=1)
                for k in range(n_f):
                    zr[wi, zi, k] = corr_1d(np.nanmean(X[:, m, k], axis=1), yz)

            best = int(np.nanargmax(med_win[wi]))
            rows.append((wname, lo, hi, n_leads, np.nanmedian(med_win[wi]),
                         med_win[wi][best], feature_vars[best]))
            print(f"    {wname:>8} (d{lo}-{hi}, {n_leads:>2} leads): "
                  f"median |r| {np.nanmedian(med_win[wi]):.3f}, "
                  f"best {med_win[wi][best]:.3f} ({feature_vars[best]})", flush=True)
            del X, y

    with stage(f"Single-day reference (lead {COMPARE_DAILY_LEAD})"):
        Xd, yd, dd, _ = window_slice(coarse, imd_cells, COMPARE_DAILY_LEAD,
                                     COMPARE_DAILY_LEAD, lat_pts, lon_pts,
                                     feature_vars)
        co = _clim_grid(yd[tr_i], dd[tr_i], CLIM_WINDOW_DAYS)
        yd = yd - co[dd - 1]
        for k in range(n_f):
            cm = _clim_grid(Xd[tr_i, :, k], dd[tr_i], CLIM_WINDOW_DAYS)
            Xd[:, :, k] -= cm[dd - 1]
        med_daily = np.nanmedian(np.abs(corr_cells(Xd, yd)), axis=0)

    with stage("Figures"):
        lift_plot(med_win, med_daily, wnames, feature_vars, OUTDIR,
                  COMPARE_DAILY_LEAD)
        zone_heatmap(zr, zone_names, feature_vars, wnames, OUTDIR)
        xr.Dataset(
            {"median_abs_r": (("window", "var"), med_win),
             "zone_r": (("window", "zone", "var"), zr),
             "median_abs_r_daily": (("var",), med_daily)},
            coords={"window": wnames, "var": feature_vars, "zone": zone_names},
        ).to_netcdf(os.path.join(OUTDIR, "correlations_windowed.nc"))

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s\n")
    print(f"{'window':>8} {'days':>8} {'leads':>6} {'med|r|':>7} {'best|r|':>8}  best var")
    for w, lo, hi, nl, med, bst, bv in rows:
        print(f"{w:>8} {f'{lo}-{hi}':>8} {nl:>6} {med:>7.3f} {bst:>8.3f}  {bv}")
    print(f"\nsingle day (lead {COMPARE_DAILY_LEAD}): median |r| "
          f"{np.nanmedian(med_daily):.3f}, best {med_daily.max():.3f}")
    print("\n-> windowed_vs_daily.png is the plot that answers the question.")
    print("   If week3-4 sits on top of the single-day line, windowing bought")
    print("   nothing and the collapse is real, not a noise artefact.")

NameError: name 'ds_ecmv' is not defined

In [ ]:
"""
RAW-VALUE correlation diagnostics. No climatology, no anomalies, nothing
subtracted from anything.

  1. Per-lead box plots: Pearson r between each predictor and rain, computed
     per cell, distribution shown across the ~4964 valid IMD cells.
  2. Four-zone stratification + inter-zone rain correlation matrix.

Read as: "how much do these fields co-vary with rain, as they come out of the
archive". The seasonal cycle is in both sides, so expect high r almost
everywhere -- that is what raw correlation measures and it is a legitimate
first look at the data. Just don't read these numbers as forecast skill.


ZONES
-----
IMD's four homogeneous regions (NWI, CI, NEI, SPIN) are groups of
meteorological subdivisions (Parthasarathy et al. 1995), not lat/lon boxes.
Zheng et al. (2016, JGR 10.1002/2016JD025135) note gridded data "can be
projected only approximately onto the four IMD regions". ZONES_APPROX below
is therefore an approximation -- caption it as such, or set ZONE_SHAPEFILE
to IMD's subdivision file for the correct route. Check zone_map.png.
"""

import os
import time
from contextlib import contextmanager

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
DTYPE = np.float32
COARSE_PAD = 2.0
LEADS = None                    # e.g. range(14, 43); None = all
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS only
OUTDIR = "../results/diagnostics/corr_3_day"

LAT_CUT, LON_CUT = 23.5, 82.5

ZONES_APPROX = [
    ("NW", lambda la, lo: (la >= LAT_CUT) & (lo <  LON_CUT)),
    ("NE", lambda la, lo: (la >= LAT_CUT) & (lo >= LON_CUT)),
    ("SW", lambda la, lo: (la <  LAT_CUT) & (lo <  LON_CUT)),
    ("SE", lambda la, lo: (la <  LAT_CUT) & (lo >= LON_CUT)),
]


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- plumbing ----------------

def normalize_step(ds, verbose=True):
    """zarr round-trips lose timedelta encoding, so step often comes back as
    a bare int (days or hours). Coerce either way."""
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def build_cells(imd_ds, var=IMD_TARGET_VAR):
    """IMD's valid cells. Not a land-sea mask -- it's the gauge-network
    footprint, so Pakistan/Nepal/Bangladesh are NaN despite being land."""
    mask2d = imd_ds[var].notnull().any(dim="time").compute()
    stacked = mask2d.stack(cell=("lat", "lon"))
    cells = stacked[stacked.values]
    print(f"    {len(cells['cell'])}/{mask2d.size} valid cells")
    return cells


def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, leads=LEADS, pad=COARSE_PAD):
    """ECMWF over India's box at native coarse res, into RAM. One disk pass;
    the 0.25 expansion happens per lead and is discarded."""
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)   # slice() returns empty on descending
    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad),
                       lon=slice(lon0 - pad, lon1 + pad))
    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    if leads is not None:
        sub = sub.isel(step=list(leads))
    print(f"    {dict(sub.sizes)} x {len(sub.data_vars)} vars")
    with ProgressBar():
        return sub.astype(DTYPE).compute()


def assign_zones(cells, shapefile=ZONE_SHAPEFILE):
    lat, lon = cells["lat"].values, cells["lon"].values
    if shapefile is not None:
        import geopandas as gpd
        from shapely.geometry import Point
        gdf = gpd.read_file(shapefile)
        pts = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(lon, lat)],
                               crs=gdf.crs)
        joined = gpd.sjoin(pts, gdf, how="left", predicate="within")
        # map subdivision names -> NWI/CI/NEI/SPIN per Parthasarathy 1995 Table 1
        return joined.iloc[:, -1].values
    z = np.empty(len(lat), dtype=object)
    un = np.ones(len(lat), bool)
    for name, rule in ZONES_APPROX:
        m = rule(lat, lon) & un
        z[m] = name
        un &= ~m
    return z


# ---------------- correlation ----------------

def corr_cells(X, y):
    """X (n, c, f), y (n, c) -> (c, f) Pearson r per cell per variable.
    Vectorised; matches np.corrcoef to ~1e-6. NaN-safe."""
    Xc = X - np.nanmean(X, axis=0)
    yc = y - np.nanmean(y, axis=0)
    num = np.nansum(Xc * yc[:, :, None], axis=0)
    den = np.sqrt(np.nansum(Xc ** 2, axis=0) * np.nansum(yc ** 2, axis=0)[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def corr_1d(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    a, b = x[m] - x[m].mean(), y[m] - y[m].mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ---------------- plots ----------------

def boxplot_lead(r, feature_vars, lead, outdir):
    order = np.argsort(-np.nanmedian(np.abs(r), axis=0))
    data = [r[:, k][np.isfinite(r[:, k])] for k in order]
    labels = [feature_vars[k] for k in order]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, whis=(5, 95))
    for p in bp["boxes"]:
        p.set_facecolor("#4C78A8")
        p.set_alpha(0.65)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Pearson r vs rain (raw values, per cell)")
    ax.set_ylim(-1, 1)
    ax.set_title(f"Lead {lead} d  --  RAW values, distribution across "
                 f"{len(data[0])} cells, sorted by |median r|")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"corr_box_lead{lead:02d}.png"), dpi=110)
    plt.close(fig)


def summary_plot(med, feature_vars, leads, outdir):
    """median |r| vs lead, one line per variable. Easier to read than 43
    separate box plots; use the boxes to drill into a specific lead."""
    fig, ax = plt.subplots(figsize=(11, 6))
    final = np.nanmean(med[-5:], axis=0)
    for k in np.argsort(-final):
        ax.plot(leads, med[:, k], lw=1.4, label=feature_vars[k])
    ax.set_xlabel("lead (days)")
    ax.set_ylabel("median |r| across cells")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=2, loc="lower left")
    ax.set_title("Raw predictor-rain correlation by lead")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "summary_corr_vs_lead.png"), dpi=130)
    plt.close(fig)


def zone_heatmaps(zr, zones, feature_vars, leads, outdir):
    fig, axes = plt.subplots(1, len(zones), figsize=(5.5 * len(zones), 7), sharey=True)
    axes = np.atleast_1d(axes)
    for zi, (ax, zname) in enumerate(zip(axes, zones)):
        im = ax.imshow(zr[:, zi, :].T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1,
                       extent=[leads[0], leads[-1], len(feature_vars) - 0.5, -0.5])
        ax.set_yticks(range(len(feature_vars)))
        ax.set_yticklabels(feature_vars, fontsize=7)
        ax.set_xlabel("lead (days)")
        ax.set_title(zname)
    fig.colorbar(im, ax=axes, label="r (zone-mean predictor vs zone-mean rain, raw)")
    fig.savefig(os.path.join(outdir, "zone_corr_heatmap.png"), dpi=130,
                bbox_inches="tight")
    plt.close(fig)


def zone_map(cells, zlab, zones, outdir):
    """Sanity check. Box zones have straight edges cutting through states."""
    da = xr.DataArray(np.array([zones.index(z) for z in zlab], float),
                      coords={"cell": cells["cell"]}, dims="cell").unstack("cell")
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.pcolormesh(da["lon"], da["lat"], da.values, cmap="tab10", vmin=0, vmax=9)
    cb = fig.colorbar(im, ax=ax, ticks=range(len(zones)))
    cb.ax.set_yticklabels(zones)
    ax.set_title("Zone assignment (approximate, not IMD polygons)")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "zone_map.png"), dpi=130)
    plt.close(fig)


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_new   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    os.makedirs(OUTDIR, exist_ok=True)
    t0 = time.perf_counter()

    with stage("Setup"):
        cells = build_cells(imd_ds)
        lat_pts = xr.DataArray(cells["lat"].values, dims="cell")
        lon_pts = xr.DataArray(cells["lon"].values, dims="cell")
        zlab = assign_zones(cells)
        zones = [z for z, _ in ZONES_APPROX] if ZONE_SHAPEFILE is None \
            else sorted(set(zlab))
        for z in zones:
            print(f"    zone {z}: {(zlab == z).sum()} cells")

    with stage("Loading coarse ECMWF"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)
        leads = (coarse["step"].values / np.timedelta64(1, "D")).astype(int)

    with stage("Loading IMD"):
        imd_cells = imd_ds[IMD_TARGET_VAR].stack(cell=("lat", "lon")).sel(
            cell=cells["cell"]).astype(DTYPE)
        with ProgressBar():
            imd_cells = imd_cells.compute()
        imd_cells = imd_cells.assign_coords(time=imd_cells["time"].dt.floor("D"))

    n_z, n_f = len(zones), len(feature_vars)
    med = np.full((len(leads), n_f), np.nan)
    zr = np.full((len(leads), n_z, n_f), np.nan)
    zone_rain = np.full((len(leads), n_z, coarse.sizes["time"]), np.nan)

    with stage(f"Correlations: {len(leads)} leads x {n_f} vars"):
        for j, lead in enumerate(leads):
            s = coarse.isel(step=j)
            # interp straight to the valid cells -- the 129x135 rectangle is
            # never allocated, so we skip 71.5% of the grid
            X = np.stack([s[v].interp(lat=lat_pts, lon=lon_pts).values
                          for v in feature_vars], axis=-1).astype(DTYPE)
            vt = s["valid_time"].dt.floor("D").values
            y = imd_cells.reindex(time=vt).values

            r = corr_cells(X, y)                       # (c, f)
            med[j] = np.nanmedian(np.abs(r), axis=0)
            boxplot_lead(r, feature_vars, int(lead), OUTDIR)

            for zi, z in enumerate(zones):
                m = (zlab == z)
                yz = np.nanmean(y[:, m], axis=1)
                zone_rain[j, zi] = yz
                for k in range(n_f):
                    zr[j, zi, k] = corr_1d(np.nanmean(X[:, m, k], axis=1), yz)

            del X, y
            print(f"    lead {int(lead):>2} done", flush=True)

    with stage("Summary figures"):
        summary_plot(med, feature_vars, leads, OUTDIR)
        zone_heatmaps(zr, zones, feature_vars, leads, OUTDIR)
        zone_map(cells, zlab, zones, OUTDIR)

        print(f"\n    Inter-zone rain correlation (raw, lead {leads[-1]}):")
        print("         " + "".join(f"{z:>7}" for z in zones))
        M = np.eye(n_z)
        for a in range(n_z):
            row = [corr_1d(zone_rain[-1, a], zone_rain[-1, b]) for b in range(n_z)]
            M[a] = row
            print(f"    {zones[a]:>5}" + "".join(f"{v:>7.2f}" for v in row))
        np.save(os.path.join(OUTDIR, "interzone_corr.npy"), M)

        xr.Dataset(
            {"median_abs_r": (("lead", "var"), med),
             "zone_r": (("lead", "zone", "var"), zr)},
            coords={"lead": leads, "var": feature_vars, "zone": zones},
        ).to_netcdf(os.path.join(OUTDIR, "correlations_raw.nc"))

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s")
    print(f"{len(leads)} box plots + summary + zone heatmap + zone map -> {OUTDIR}/")


NameError: name 'ZONE_SHAPEFILE' is not defined

In [ ]:
"""
RAW-VALUE correlation diagnostics. No climatology, no anomalies, nothing
subtracted from anything.

  1. Per-lead box plots: Pearson r between each predictor and rain, computed
     per cell, distribution shown across the ~4964 valid IMD cells.
  2. Four-zone stratification + inter-zone rain correlation matrix.

Read as: "how much do these fields co-vary with rain, as they come out of the
archive". The seasonal cycle is in both sides, so expect high r almost
everywhere -- that is what raw correlation measures and it is a legitimate
first look at the data. Just don't read these numbers as forecast skill.


ZONES
-----
IMD's four homogeneous regions (NWI, CI, NEI, SPIN) are groups of
meteorological subdivisions (Parthasarathy et al. 1995), not lat/lon boxes.
Zheng et al. (2016, JGR 10.1002/2016JD025135) note gridded data "can be
projected only approximately onto the four IMD regions". ZONES_APPROX below
is therefore an approximation -- caption it as such, or set ZONE_SHAPEFILE
to IMD's subdivision file for the correct route. Check zone_map.png.
"""

import os
import time
from contextlib import contextmanager

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
DTYPE = np.float32
COARSE_PAD = 2.0
LEADS = None                    # e.g. range(14, 43); None = all
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS only
OUTDIR = "../results/diagnostics/corr_raw"

LAT_CUT, LON_CUT = 23.5, 82.5

ZONES_APPROX = [
    ("NW", lambda la, lo: (la >= LAT_CUT) & (lo <  LON_CUT)),
    ("NE", lambda la, lo: (la >= LAT_CUT) & (lo >= LON_CUT)),
    ("SW", lambda la, lo: (la <  LAT_CUT) & (lo <  LON_CUT)),
    ("SE", lambda la, lo: (la <  LAT_CUT) & (lo >= LON_CUT)),
]


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- plumbing ----------------

def normalize_step(ds, verbose=True):
    """zarr round-trips lose timedelta encoding, so step often comes back as
    a bare int (days or hours). Coerce either way."""
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def build_cells(imd_ds, var=IMD_TARGET_VAR):
    """IMD's valid cells. Not a land-sea mask -- it's the gauge-network
    footprint, so Pakistan/Nepal/Bangladesh are NaN despite being land."""
    mask2d = imd_ds[var].notnull().any(dim="time").compute()
    stacked = mask2d.stack(cell=("lat", "lon"))
    cells = stacked[stacked.values]
    print(f"    {len(cells['cell'])}/{mask2d.size} valid cells")
    return cells


def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, leads=LEADS, pad=COARSE_PAD):
    """ECMWF over India's box at native coarse res, into RAM. One disk pass;
    the 0.25 expansion happens per lead and is discarded."""
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)   # slice() returns empty on descending
    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad),
                       lon=slice(lon0 - pad, lon1 + pad))
    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    if leads is not None:
        sub = sub.isel(step=list(leads))
    print(f"    {dict(sub.sizes)} x {len(sub.data_vars)} vars")
    with ProgressBar():
        return sub.astype(DTYPE).compute()


def assign_zones(cells, shapefile=ZONE_SHAPEFILE):
    lat, lon = cells["lat"].values, cells["lon"].values
    if shapefile is not None:
        import geopandas as gpd
        from shapely.geometry import Point
        gdf = gpd.read_file(shapefile)
        pts = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(lon, lat)],
                               crs=gdf.crs)
        joined = gpd.sjoin(pts, gdf, how="left", predicate="within")
        # map subdivision names -> NWI/CI/NEI/SPIN per Parthasarathy 1995 Table 1
        return joined.iloc[:, -1].values
    z = np.empty(len(lat), dtype=object)
    un = np.ones(len(lat), bool)
    for name, rule in ZONES_APPROX:
        m = rule(lat, lon) & un
        z[m] = name
        un &= ~m
    return z


# ---------------- correlation ----------------

def corr_cells(X, y):
    """X (n, c, f), y (n, c) -> (c, f) Pearson r per cell per variable.
    Vectorised; matches np.corrcoef to ~1e-6. NaN-safe."""
    Xc = X - np.nanmean(X, axis=0)
    yc = y - np.nanmean(y, axis=0)
    num = np.nansum(Xc * yc[:, :, None], axis=0)
    den = np.sqrt(np.nansum(Xc ** 2, axis=0) * np.nansum(yc ** 2, axis=0)[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def corr_1d(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    a, b = x[m] - x[m].mean(), y[m] - y[m].mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ---------------- plots ----------------

def boxplot_lead(r, feature_vars, lead, outdir):
    order = np.argsort(-np.nanmedian(np.abs(r), axis=0))
    data = [r[:, k][np.isfinite(r[:, k])] for k in order]
    labels = [feature_vars[k] for k in order]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, whis=(5, 95))
    for p in bp["boxes"]:
        p.set_facecolor("#4C78A8")
        p.set_alpha(0.65)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Pearson r vs rain (raw values, per cell)")
    ax.set_ylim(-1, 1)
    ax.set_title(f"Lead {lead} d  --  RAW values, distribution across "
                 f"{len(data[0])} cells, sorted by |median r|")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"corr_box_lead{lead:02d}.png"), dpi=110)
    plt.close(fig)


def summary_plot(med, feature_vars, leads, outdir):
    """median |r| vs lead, one line per variable. Easier to read than 43
    separate box plots; use the boxes to drill into a specific lead."""
    fig, ax = plt.subplots(figsize=(11, 6))
    final = np.nanmean(med[-5:], axis=0)
    for k in np.argsort(-final):
        ax.plot(leads, med[:, k], lw=1.4, label=feature_vars[k])
    ax.set_xlabel("lead (days)")
    ax.set_ylabel("median |r| across cells")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=2, loc="lower left")
    ax.set_title("Raw predictor-rain correlation by lead")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "summary_corr_vs_lead.png"), dpi=130)
    plt.close(fig)


def zone_heatmaps(zr, zones, feature_vars, leads, outdir):
    fig, axes = plt.subplots(1, len(zones), figsize=(5.5 * len(zones), 7), sharey=True)
    axes = np.atleast_1d(axes)
    for zi, (ax, zname) in enumerate(zip(axes, zones)):
        im = ax.imshow(zr[:, zi, :].T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1,
                       extent=[leads[0], leads[-1], len(feature_vars) - 0.5, -0.5])
        ax.set_yticks(range(len(feature_vars)))
        ax.set_yticklabels(feature_vars, fontsize=7)
        ax.set_xlabel("lead (days)")
        ax.set_title(zname)
    fig.colorbar(im, ax=axes, label="r (zone-mean predictor vs zone-mean rain, raw)")
    fig.savefig(os.path.join(outdir, "zone_corr_heatmap.png"), dpi=130,
                bbox_inches="tight")
    plt.close(fig)


def zone_map(cells, zlab, zones, outdir):
    """Sanity check. Box zones have straight edges cutting through states."""
    da = xr.DataArray(np.array([zones.index(z) for z in zlab], float),
                      coords={"cell": cells["cell"]}, dims="cell").unstack("cell")
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.pcolormesh(da["lon"], da["lat"], da.values, cmap="tab10", vmin=0, vmax=9)
    cb = fig.colorbar(im, ax=ax, ticks=range(len(zones)))
    cb.ax.set_yticklabels(zones)
    ax.set_title("Zone assignment (approximate, not IMD polygons)")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "zone_map.png"), dpi=130)
    plt.close(fig)


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_new   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    os.makedirs(OUTDIR, exist_ok=True)
    t0 = time.perf_counter()

    with stage("Setup"):
        cells = build_cells(imd_ds)
        lat_pts = xr.DataArray(cells["lat"].values, dims="cell")
        lon_pts = xr.DataArray(cells["lon"].values, dims="cell")
        zlab = assign_zones(cells)
        zones = [z for z, _ in ZONES_APPROX] if ZONE_SHAPEFILE is None \
            else sorted(set(zlab))
        for z in zones:
            print(f"    zone {z}: {(zlab == z).sum()} cells")

    with stage("Loading coarse ECMWF"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)
        leads = (coarse["step"].values / np.timedelta64(1, "D")).astype(int)

    with stage("Loading IMD"):
        imd_cells = imd_ds[IMD_TARGET_VAR].stack(cell=("lat", "lon")).sel(
            cell=cells["cell"]).astype(DTYPE)
        with ProgressBar():
            imd_cells = imd_cells.compute()
        imd_cells = imd_cells.assign_coords(time=imd_cells["time"].dt.floor("D"))

    n_z, n_f = len(zones), len(feature_vars)
    med = np.full((len(leads), n_f), np.nan)
    zr = np.full((len(leads), n_z, n_f), np.nan)
    zone_rain = np.full((len(leads), n_z, coarse.sizes["time"]), np.nan)

    with stage(f"Correlations: {len(leads)} leads x {n_f} vars"):
        for j, lead in enumerate(leads):
            s = coarse.isel(step=j)
            # interp straight to the valid cells -- the 129x135 rectangle is
            # never allocated, so we skip 71.5% of the grid
            X = np.stack([s[v].interp(lat=lat_pts, lon=lon_pts).values
                          for v in feature_vars], axis=-1).astype(DTYPE)
            vt = s["valid_time"].dt.floor("D").values
            y = imd_cells.reindex(time=vt).values

            r = corr_cells(X, y)                       # (c, f)
            med[j] = np.nanmedian(np.abs(r), axis=0)
            boxplot_lead(r, feature_vars, int(lead), OUTDIR)

            for zi, z in enumerate(zones):
                m = (zlab == z)
                yz = np.nanmean(y[:, m], axis=1)
                zone_rain[j, zi] = yz
                for k in range(n_f):
                    zr[j, zi, k] = corr_1d(np.nanmean(X[:, m, k], axis=1), yz)

            del X, y
            print(f"    lead {int(lead):>2} done", flush=True)

    with stage("Summary figures"):
        summary_plot(med, feature_vars, leads, OUTDIR)
        zone_heatmaps(zr, zones, feature_vars, leads, OUTDIR)
        zone_map(cells, zlab, zones, OUTDIR)

        print(f"\n    Inter-zone rain correlation (raw, lead {leads[-1]}):")
        print("         " + "".join(f"{z:>7}" for z in zones))
        M = np.eye(n_z)
        for a in range(n_z):
            row = [corr_1d(zone_rain[-1, a], zone_rain[-1, b]) for b in range(n_z)]
            M[a] = row
            print(f"    {zones[a]:>5}" + "".join(f"{v:>7.2f}" for v in row))
        np.save(os.path.join(OUTDIR, "interzone_corr.npy"), M)

        xr.Dataset(
            {"median_abs_r": (("lead", "var"), med),
             "zone_r": (("lead", "zone", "var"), zr)},
            coords={"lead": leads, "var": feature_vars, "zone": zones},
        ).to_netcdf(os.path.join(OUTDIR, "correlations_raw.nc"))

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s")
    print(f"{len(leads)} box plots + summary + zone heatmap + zone map -> {OUTDIR}/")


NameError: name 'ZONE_SHAPEFILE' is not defined

In [7]:
"""
RAW-VALUE correlation diagnostics. No climatology, no anomalies, nothing
subtracted from anything.

  1. Per-lead box plots: Pearson r between each predictor and rain, computed
     per cell, distribution shown across the ~4964 valid IMD cells.
  2. Four-zone stratification + inter-zone rain correlation matrix.

Read as: "how much do these fields co-vary with rain, as they come out of the
archive". The seasonal cycle is in both sides, so expect high r almost
everywhere -- that is what raw correlation measures and it is a legitimate
first look at the data. Just don't read these numbers as forecast skill.


ZONES
-----
IMD's four homogeneous regions (NWI, CI, NEI, SPIN) are groups of
meteorological subdivisions (Parthasarathy et al. 1995), not lat/lon boxes.
Zheng et al. (2016, JGR 10.1002/2016JD025135) note gridded data "can be
projected only approximately onto the four IMD regions". ZONES_APPROX below
is therefore an approximation -- caption it as such, or set ZONE_SHAPEFILE
to IMD's subdivision file for the correct route. Check zone_map.png.
"""

import os
import time
from contextlib import contextmanager

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"
DTYPE = np.float32
COARSE_PAD = 2.0
LEADS = None                    # e.g. range(14, 43); None = all
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS only
OUTDIR = "../results/diagnostics/corr_raw"

LAT_CUT, LON_CUT = 23.5, 82.5

ZONES_APPROX = [
    ("NW", lambda la, lo: (la >= LAT_CUT) & (lo <  LON_CUT)),
    ("NE", lambda la, lo: (la >= LAT_CUT) & (lo >= LON_CUT)),
    ("SW", lambda la, lo: (la <  LAT_CUT) & (lo <  LON_CUT)),
    ("SE", lambda la, lo: (la <  LAT_CUT) & (lo >= LON_CUT)),
]


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ---------------- plumbing ----------------

def normalize_step(ds, verbose=True):
    """zarr round-trips lose timedelta encoding, so step often comes back as
    a bare int (days or hours). Coerce either way."""
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds, verbose=True):
    ds = normalize_step(ds, verbose=verbose)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        if verbose:
            print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def build_cells(imd_ds, var=IMD_TARGET_VAR):
    """IMD's valid cells. Not a land-sea mask -- it's the gauge-network
    footprint, so Pakistan/Nepal/Bangladesh are NaN despite being land."""
    mask2d = imd_ds[var].notnull().any(dim="time").compute()
    stacked = mask2d.stack(cell=("lat", "lon"))
    cells = stacked[stacked.values]
    print(f"    {len(cells['cell'])}/{mask2d.size} valid cells")
    return cells


def load_coarse(ecmwf_ds, imd_ds, months=MONTHS, leads=LEADS, pad=COARSE_PAD):
    """ECMWF over India's box at native coarse res, into RAM. One disk pass;
    the 0.25 expansion happens per lead and is discarded."""
    ecmwf_ds = ensure_valid_time(ecmwf_ds)
    for c in ("lat", "lon"):
        if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
            ecmwf_ds = ecmwf_ds.sortby(c)   # slice() returns empty on descending
    lat0, lat1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
    lon0, lon1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
    sub = ecmwf_ds.sel(lat=slice(lat0 - pad, lat1 + pad),
                       lon=slice(lon0 - pad, lon1 + pad))
    if months is not None:
        sub = sub.sel(time=sub["time"].dt.month.isin(list(months)))
    if leads is not None:
        sub = sub.isel(step=list(leads))
    print(f"    {dict(sub.sizes)} x {len(sub.data_vars)} vars")
    with ProgressBar():
        return sub.astype(DTYPE).compute()


def assign_zones(cells, shapefile=ZONE_SHAPEFILE):
    lat, lon = cells["lat"].values, cells["lon"].values
    if shapefile is not None:
        import geopandas as gpd
        from shapely.geometry import Point
        gdf = gpd.read_file(shapefile)
        pts = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(lon, lat)],
                               crs=gdf.crs)
        joined = gpd.sjoin(pts, gdf, how="left", predicate="within")
        # map subdivision names -> NWI/CI/NEI/SPIN per Parthasarathy 1995 Table 1
        return joined.iloc[:, -1].values
    z = np.empty(len(lat), dtype=object)
    un = np.ones(len(lat), bool)
    for name, rule in ZONES_APPROX:
        m = rule(lat, lon) & un
        z[m] = name
        un &= ~m
    return z


# ---------------- correlation ----------------

def corr_cells(X, y):
    """X (n, c, f), y (n, c) -> (c, f) Pearson r per cell per variable.
    Vectorised; matches np.corrcoef to ~1e-6. NaN-safe."""
    Xc = X - np.nanmean(X, axis=0)
    yc = y - np.nanmean(y, axis=0)
    num = np.nansum(Xc * yc[:, :, None], axis=0)
    den = np.sqrt(np.nansum(Xc ** 2, axis=0) * np.nansum(yc ** 2, axis=0)[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)


def corr_1d(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    a, b = x[m] - x[m].mean(), y[m] - y[m].mean()
    d = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


# ---------------- plots ----------------

def boxplot_lead(r, feature_vars, lead, outdir):
    order = np.argsort(-np.nanmedian(np.abs(r), axis=0))
    data = [r[:, k][np.isfinite(r[:, k])] for k in order]
    labels = [feature_vars[k] for k in order]

    fig, ax = plt.subplots(figsize=(13, 6))
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, whis=(5, 95))
    for p in bp["boxes"]:
        p.set_facecolor("#4C78A8")
        p.set_alpha(0.65)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Pearson r vs rain (raw values, per cell)")
    ax.set_ylim(-1, 1)
    ax.set_title(f"Lead {lead} d  --  RAW values, distribution across "
                 f"{len(data[0])} cells, sorted by |median r|")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, f"corr_box_lead{lead:02d}.png"), dpi=110)
    plt.close(fig)


def summary_plot(med, feature_vars, leads, outdir):
    """median |r| vs lead, one line per variable. Easier to read than 43
    separate box plots; use the boxes to drill into a specific lead."""
    fig, ax = plt.subplots(figsize=(11, 6))
    final = np.nanmean(med[-5:], axis=0)
    for k in np.argsort(-final):
        ax.plot(leads, med[:, k], lw=1.4, label=feature_vars[k])
    ax.set_xlabel("lead (days)")
    ax.set_ylabel("median |r| across cells")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=2, loc="lower left")
    ax.set_title("Raw predictor-rain correlation by lead")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "summary_corr_vs_lead.png"), dpi=130)
    plt.close(fig)


def zone_heatmaps(zr, zones, feature_vars, leads, outdir):
    fig, axes = plt.subplots(1, len(zones), figsize=(5.5 * len(zones), 7), sharey=True)
    axes = np.atleast_1d(axes)
    for zi, (ax, zname) in enumerate(zip(axes, zones)):
        im = ax.imshow(zr[:, zi, :].T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1,
                       extent=[leads[0], leads[-1], len(feature_vars) - 0.5, -0.5])
        ax.set_yticks(range(len(feature_vars)))
        ax.set_yticklabels(feature_vars, fontsize=7)
        ax.set_xlabel("lead (days)")
        ax.set_title(zname)
    fig.colorbar(im, ax=axes, label="r (zone-mean predictor vs zone-mean rain, raw)")
    fig.savefig(os.path.join(outdir, "zone_corr_heatmap.png"), dpi=130,
                bbox_inches="tight")
    plt.close(fig)


def zone_map(cells, zlab, zones, outdir):
    """Sanity check. Box zones have straight edges cutting through states."""
    da = xr.DataArray(np.array([zones.index(z) for z in zlab], float),
                      coords={"cell": cells["cell"]}, dims="cell").unstack("cell")
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.pcolormesh(da["lon"], da["lat"], da.values, cmap="tab10", vmin=0, vmax=9)
    cb = fig.colorbar(im, ax=ax, ticks=range(len(zones)))
    cb.ax.set_yticklabels(zones)
    ax.set_title("Zone assignment (approximate, not IMD polygons)")
    ax.set_xlabel("lon"); ax.set_ylabel("lat")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "zone_map.png"), dpi=130)
    plt.close(fig)


# ---------------- driver ----------------

if __name__ == "__main__":
    ecmwf_ds = ds_ecmv   # noqa: F821
    imd_ds = ds_imd      # noqa: F821

    os.makedirs(OUTDIR, exist_ok=True)
    t0 = time.perf_counter()

    with stage("Setup"):
        cells = build_cells(imd_ds)
        lat_pts = xr.DataArray(cells["lat"].values, dims="cell")
        lon_pts = xr.DataArray(cells["lon"].values, dims="cell")
        zlab = assign_zones(cells)
        zones = [z for z, _ in ZONES_APPROX] if ZONE_SHAPEFILE is None \
            else sorted(set(zlab))
        for z in zones:
            print(f"    zone {z}: {(zlab == z).sum()} cells")

    with stage("Loading coarse ECMWF"):
        coarse = load_coarse(ecmwf_ds, imd_ds)
        feature_vars = list(coarse.data_vars)
        leads = (coarse["step"].values / np.timedelta64(1, "D")).astype(int)

    with stage("Loading IMD"):
        imd_cells = imd_ds[IMD_TARGET_VAR].stack(cell=("lat", "lon")).sel(
            cell=cells["cell"]).astype(DTYPE)
        with ProgressBar():
            imd_cells = imd_cells.compute()
        imd_cells = imd_cells.assign_coords(time=imd_cells["time"].dt.floor("D"))

    n_z, n_f = len(zones), len(feature_vars)
    med = np.full((len(leads), n_f), np.nan)
    zr = np.full((len(leads), n_z, n_f), np.nan)
    zone_rain = np.full((len(leads), n_z, coarse.sizes["time"]), np.nan)

    with stage(f"Correlations: {len(leads)} leads x {n_f} vars"):
        for j, lead in enumerate(leads):
            s = coarse.isel(step=j)
            # interp straight to the valid cells -- the 129x135 rectangle is
            # never allocated, so we skip 71.5% of the grid
            X = np.stack([s[v].interp(lat=lat_pts, lon=lon_pts).values
                          for v in feature_vars], axis=-1).astype(DTYPE)
            vt = s["valid_time"].dt.floor("D").values
            y = imd_cells.reindex(time=vt).values

            r = corr_cells(X, y)                       # (c, f)
            med[j] = np.nanmedian(np.abs(r), axis=0)
            boxplot_lead(r, feature_vars, int(lead), OUTDIR)

            for zi, z in enumerate(zones):
                m = (zlab == z)
                yz = np.nanmean(y[:, m], axis=1)
                zone_rain[j, zi] = yz
                for k in range(n_f):
                    zr[j, zi, k] = corr_1d(np.nanmean(X[:, m, k], axis=1), yz)

            del X, y
            print(f"    lead {int(lead):>2} done", flush=True)

    with stage("Summary figures"):
        summary_plot(med, feature_vars, leads, OUTDIR)
        zone_heatmaps(zr, zones, feature_vars, leads, OUTDIR)
        zone_map(cells, zlab, zones, OUTDIR)

        print(f"\n    Inter-zone rain correlation (raw, lead {leads[-1]}):")
        print("         " + "".join(f"{z:>7}" for z in zones))
        M = np.eye(n_z)
        for a in range(n_z):
            row = [corr_1d(zone_rain[-1, a], zone_rain[-1, b]) for b in range(n_z)]
            M[a] = row
            print(f"    {zones[a]:>5}" + "".join(f"{v:>7.2f}" for v in row))
        np.save(os.path.join(OUTDIR, "interzone_corr.npy"), M)

        xr.Dataset(
            {"median_abs_r": (("lead", "var"), med),
             "zone_r": (("lead", "zone", "var"), zr)},
            coords={"lead": leads, "var": feature_vars, "zone": zones},
        ).to_netcdf(os.path.join(OUTDIR, "correlations_raw.nc"))

    print(f"\nTotal: {time.perf_counter() - t0:.1f}s")
    print(f"{len(leads)} box plots + summary + zone heatmap + zone map -> {OUTDIR}/")


NameError: name 'ZONE_SHAPEFILE' is not defined